This is the CPU side with changing only associativity values (so there are 6 threads)

In [1]:
%%writefile cache_sim_flat.h
#ifndef SIM_CACHE_FLAT_H
#define SIM_CACHE_FLAT_H
#include <cmath>
#include <tuple>
#include <utility>
#include <cstdint>
#include <climits>

typedef
struct {
   uint32_t BLOCKSIZE;
   uint32_t L1_SIZE;
   uint32_t L1_ASSOC;
   uint32_t L2_SIZE;
   uint32_t L2_ASSOC;
  uint32_t PREF_N;
  uint32_t PREF_M;
} cache_params_t;


//structure for all the cache parameters
struct cache_params
{
    int read_hit = 0;
    int write_hit = 0;
    int read_miss = 0;
    int write_miss = 0;
    int writeback_to_lower_mem = 0;

    int prefetch_reads_from_l1 = 0;
    int prefetch_misses_from_l1 = 0;

    int prefetch_req = 0;  //prefetches issued

    //hits on stream buffer
    int sb_read_hits = 0;
    int sb_write_hits = 0;
};



	//for implemetation of lru policy wherein counters are maintained indicating the recency
	//0: most recently used
	void lru_policy(int l_assoc, int l_index_set, int l_index_tag, int* l_arr_lru)
  {
    for (int q = 0; q < l_assoc; ++q) {
        if (l_arr_lru[l_index_set * l_assoc + q] < l_arr_lru[l_index_set * l_assoc + l_index_tag])
            l_arr_lru[l_index_set * l_assoc + q]++;
    }
    l_arr_lru[l_index_set * l_assoc + l_index_tag] = 0;
  }


  //getting the number of sets and number of bits required for set and blockoffset
	std::tuple<int, int, int> set_tag_bits_cal(int block_size, int cache_size, int assoc)
	{
	    int no_of_sets = cache_size / (block_size * assoc);
	    int set_bits = std::log2(no_of_sets);
	    int blockoffset_bits = std::log2(block_size);
	    return {set_bits, blockoffset_bits, no_of_sets};
	}
  //parsing out the set and tag fields out of the 32 bits address
	std::pair<uint32_t, uint32_t> parseaddress(uint32_t address, int no_set_bits, int no_blockoffset_bits)
 	{
	    uint32_t index_mask = (1u << no_set_bits) - 1;
	    uint32_t set = (address >> no_blockoffset_bits) & index_mask;
	    uint32_t tag = address >> (no_blockoffset_bits + no_set_bits);
	    return {tag, set};
	}

  void sb_mark_mru(int p_index, int* sb_rec, uint32_t pref_N)
  {
    if (pref_N == 0)
    return;
    for (size_t i = 0; i < pref_N; ++i) {
        if ((int)i != p_index)
            sb_rec[i] += 1;
    }
    sb_rec[p_index] = 0;
  }
  int sb_hit(uint32_t pref_N, uint32_t pref_M, bool* sb_valid, int* sb_top, int* sb_rec, uint32_t* sb_blocks, uint32_t block_number)
  {
    if (pref_N == 0) return -1;
    int selected = -1;
    int best_rec = INT32_MAX;
    for (size_t i = 0; i < pref_N; ++i) {
        if (!sb_valid[i]) continue;
        for (uint32_t j = 0; j < pref_M; ++j) {
            uint32_t real_index = (sb_top[i] + j) % pref_M;
            if (sb_blocks[i*pref_M + real_index] == block_number) {
                if (sb_rec[i] < best_rec) {
                    selected = (int)i;
                    best_rec = sb_rec[i];
                }
                break;
            }
        }
    }
    return selected;
  }

  int sb_pos_finder(int buf_p_index, uint32_t block_number, uint32_t pref_N, uint32_t pref_M, int* sb_top, uint32_t* sb_blocks)
  {
    if (buf_p_index < 0 || (uint32_t)buf_p_index >= pref_N) return -1;
    for (uint32_t j = 0; j < pref_M; ++j) {
        uint32_t real_index = (sb_top[buf_p_index] + j) % pref_M;
        if (sb_blocks[buf_p_index*pref_M+real_index] == block_number)
            return (int)j;
    }
    return -1;
  }

  void sb_miss(int buf_p_index, uint32_t start_block, bool* sb_valid, int* sb_top, uint32_t* sb_blocks, uint32_t pref_M, uint32_t pref_N, int* sb_rec)
  {
    if (pref_N == 0) return;
    sb_valid[buf_p_index] = true;
    sb_top[buf_p_index] = 0;
    uint32_t next = start_block + 1;
    for (uint32_t i = 0; i < pref_M; ++i)
        sb_blocks[buf_p_index*pref_M + i] = next + i;
    sb_mark_mru(buf_p_index, sb_rec, pref_N);
  }
  void sb_hit_continue(int buf_p_index, int pos, uint32_t pref_M, uint32_t pref_N,
                     int* sb_top, uint32_t* sb_blocks, int* sb_rec)
  {
    if (pref_N == 0) return;
    int count =0;
    uint32_t* remaining = new uint32_t[pref_M];
    for (uint32_t i = pos + 1; i < pref_M; ++i) {
        uint32_t real_index = (sb_top[buf_p_index] + i) % pref_M;
        remaining[count++] = sb_blocks[buf_p_index * pref_M + real_index];
    }
    uint32_t last_elem_index = (sb_top[buf_p_index]+ pref_M - 1) % pref_M;
    uint32_t next_prefetch = sb_blocks[buf_p_index * pref_M + last_elem_index] + 1;
    uint32_t* newblocks = new uint32_t[pref_M];
    int nb_count = 0;
    for (int k = 0; k < count; ++k)
      newblocks[nb_count++] = remaining[k];
    while (nb_count < (int)pref_M)
      newblocks[nb_count++] = next_prefetch++;
    for (uint32_t i = 0; i < pref_M; ++i)
      sb_blocks[buf_p_index * pref_M + i] = newblocks[i];
    sb_top[buf_p_index] = 0;
    sb_mark_mru(buf_p_index, sb_rec, pref_N);
    delete[] remaining;
    delete[] newblocks;
  }
  int sb_lru(uint32_t pref_N, bool* sb_valid, int* sb_rec)
  {
    if (pref_N == 0) return -1;
    for (size_t i = 0; i < pref_N; ++i)
        if (!sb_valid[i]) return (int)i;
    int best_p_index = 0;
    int best_rec = sb_rec[0];
    for (size_t i = 1; i < pref_N; ++i) {
        if (sb_rec[i] > best_rec) {
            best_rec = sb_rec[i];
            best_p_index = (int)i;
        }
    }
    return best_p_index;
  }
  void l2_cache(int* l2_tag_storage, int* l2_arr_valid, char* l2_arr_dirty, bool* sb_valid, int* sb_top, int* sb_rec, uint32_t* sb_blocks, uint32_t pref_M, uint32_t pref_N, cache_params& l2_counters, int l2_block_size, int l2_size, int l2_assoc, uint32_t l2_addr, char l2_rw, int* l2_arr_lru, bool is_prefetch, uint32_t* l2_arr_dirty_addr)
	{
	    auto [l2_set_bits, l2_blockoffset_bits, l2_no_of_sets] = set_tag_bits_cal(l2_block_size, l2_size, l2_assoc);
	    auto [l2_tag, l2_set] = parseaddress(l2_addr, l2_set_bits, l2_blockoffset_bits);
      if (l2_rw == 'r') {
        if (is_prefetch)
        l2_counters.prefetch_reads_from_l1++;
      }
      bool l2_search_tag = false;
      int index_tag = -1;
      for (int i = 0; i < l2_assoc; ++i) {
      if (l2_arr_valid[l2_set*l2_assoc + i] == 1 && l2_tag == l2_tag_storage[l2_set *l2_assoc + i]) {
        l2_search_tag = true;
        index_tag = i;
        break;
      }
      }

      uint32_t block_number = (l2_addr >> l2_blockoffset_bits);
      int sb_hit_p_index = -1;
      if (pref_N > 0) {
      sb_hit_p_index = sb_hit(pref_N, pref_M, sb_valid, sb_top, sb_rec,sb_blocks, block_number);
      }

      if (l2_search_tag)
      {
          lru_policy(l2_assoc, l2_set, index_tag, l2_arr_lru);
          if (l2_rw == 'r') {
          if (!is_prefetch)
            l2_counters.read_hit++;
          } else {
          l2_counters.write_hit++;
          l2_arr_dirty[l2_set * l2_assoc + index_tag] = 'D';
          }
          if (sb_hit_p_index != -1) {
            int pos = sb_pos_finder(sb_hit_p_index, block_number, pref_N, pref_M, sb_top, sb_blocks);
            if (pos >= 0) {
            sb_hit_continue(sb_hit_p_index, pos, pref_M, pref_N,
              sb_top, sb_blocks, sb_rec );
              uint32_t new_count = pos + 1;
              for (uint32_t i = 0; i < new_count; i++)
                l2_counters.prefetch_req++;
            }
          }
      }
      else
{
    if (sb_hit_p_index == -1)
    {
        if (is_prefetch)
            l2_counters.prefetch_misses_from_l1++;
        else {
            if (l2_rw == 'r') l2_counters.read_miss++;
            else l2_counters.write_miss++;
        }
    }
    else
    {
        if (!is_prefetch) {
            if (l2_rw == 'r') l2_counters.sb_read_hits++;
            else l2_counters.sb_write_hits++;
        }
    }
    for (int j = 0; j < l2_assoc; ++j) {
        if (l2_arr_lru[l2_set * l2_assoc + j] == (l2_assoc - 1)) {
            if (l2_arr_dirty[l2_set * l2_assoc + j] == 'D')
                l2_counters.writeback_to_lower_mem++;
            l2_tag_storage[l2_set * l2_assoc + j] = l2_tag;
            l2_arr_dirty[l2_set * l2_assoc + j] = (l2_rw == 'w') ? 'D' : ' ';
            l2_arr_valid[l2_set * l2_assoc + j] = 1;
            lru_policy(l2_assoc, l2_set, j, l2_arr_lru);
            break;
        }
    }
    if (sb_hit_p_index == -1) {
        if (pref_N > 0 && pref_M > 0 && !is_prefetch) {
            int sbp = sb_lru(pref_N, sb_valid, sb_rec);
            if (sbp != -1) {
                sb_miss(sbp,block_number, sb_valid, sb_top, sb_blocks, pref_M, pref_N, sb_rec);
                for (uint32_t i = 0; i < pref_M; ++i)
                    l2_counters.prefetch_req++;
            }
        }
        } else {
            int pos = sb_pos_finder(sb_hit_p_index, block_number, pref_N, pref_M, sb_top, sb_blocks);
            if (pos >= 0) {
            sb_hit_continue(sb_hit_p_index, pos, pref_M, pref_N, sb_top, sb_blocks, sb_rec);
            uint32_t new_count = pos + 1;
            for (uint32_t i = 0; i < new_count; i++)
                l2_counters.prefetch_req++;
        }
    }
  }
  }

  void l1_cache(int* l1_tag_storage, char* l1_arr_dirty, int* l1_arr_valid, uint32_t* l1_arr_dirty_addr, int* l1_arr_lru,
              int* l2_tag_storage, char* l2_arr_dirty, int* l2_arr_valid, uint32_t* l2_arr_dirty_addr,
              int* l2_arr_lru, cache_params& l1_counters, cache_params& l2_counters,
              int cache_block_size, int cache_size, int c_assoc, uint32_t l1_addr, char rw,
              int l2_size, int l2_assoc, bool* sb_valid, int* sb_top, int* sb_rec, uint32_t* sb_blocks,
              uint32_t l1_pref_N, uint32_t l1_pref_M, uint32_t l2_pref_N, uint32_t l2_pref_M)
{
    auto [l1_set_bits, l1_blockoffset_bits, l1_no_of_sets] = set_tag_bits_cal(cache_block_size, cache_size, c_assoc);
    auto [c_tag, c_set] = parseaddress(l1_addr, l1_set_bits, l1_blockoffset_bits);

    bool l1_search_tag = false;
    int index_tag = -1;
    for (int i = 0; i < c_assoc; ++i) {
        if (l1_arr_valid[c_set * c_assoc + i] == 1 && c_tag == l1_tag_storage[c_set * c_assoc + i]) {
            l1_search_tag = true;
            index_tag = i;
            break;
        }
    }

    uint32_t block_number = (l1_addr >> l1_blockoffset_bits);
    int sb_hit_p_index = -1;
    if (l1_pref_N > 0) {
        sb_hit_p_index = sb_hit(l1_pref_N, l1_pref_M, sb_valid, sb_top, sb_rec, sb_blocks, block_number);
    }

    if (l1_search_tag)
    {
        // ---- cache hit ----
        lru_policy(c_assoc, c_set, index_tag, l1_arr_lru);
        if (rw == 'r') {
            l1_counters.read_hit++;
        } else {
            l1_counters.write_hit++;
            l1_arr_dirty[c_set * c_assoc + index_tag] = 'D';
            l1_arr_dirty_addr[c_set * c_assoc + index_tag] = l1_addr;
        }

        // hit + stream buffer hit
        if (sb_hit_p_index != -1) {
            int pos = sb_pos_finder(sb_hit_p_index, block_number, l1_pref_N, l1_pref_M, sb_top, sb_blocks);
            if (pos >= 0) {
                sb_hit_continue(sb_hit_p_index, pos, l1_pref_M, l1_pref_N, sb_top, sb_blocks, sb_rec);
                uint32_t new_count = pos + 1;
                uint32_t tail_start = (sb_top[sb_hit_p_index] + l1_pref_M - new_count) % l1_pref_M;
                for (uint32_t i = 0; i < new_count; i++) {
                    uint32_t real_index = (tail_start + i) % l1_pref_M;
                    uint32_t blk = sb_blocks[sb_hit_p_index * l1_pref_M + real_index];
                    uint32_t prefetch_addr = blk << l1_blockoffset_bits;
                    l1_counters.prefetch_req++;
                    if (l2_size != 0)
                        l2_cache(l2_tag_storage, l2_arr_valid, l2_arr_dirty, sb_valid, sb_top, sb_rec,
                                 sb_blocks, l2_pref_M, l2_pref_N, l2_counters, cache_block_size, l2_size,
                                 l2_assoc, prefetch_addr, 'r', l2_arr_lru, true, l2_arr_dirty_addr);
                }
            }
        }
    }
    else
    {
        // ---- cache miss ----
        if (sb_hit_p_index == -1)
        {
            // miss + stream buffer miss
            if (rw == 'r') l1_counters.read_miss++;
            else l1_counters.write_miss++;

            for (int j = 0; j < c_assoc; ++j) {
                if (l1_arr_lru[c_set * c_assoc + j] == (c_assoc - 1)) {
                    if (l1_arr_dirty[c_set * c_assoc + j] == 'D') {
                        l1_counters.writeback_to_lower_mem++;
                        if (l2_size != 0)
                            l2_cache(l2_tag_storage, l2_arr_valid, l2_arr_dirty, sb_valid, sb_top, sb_rec,
                                     sb_blocks, l2_pref_M, l2_pref_N, l2_counters, cache_block_size, l2_size,
                                     l2_assoc, l1_arr_dirty_addr[c_set * c_assoc + j], 'w', l2_arr_lru, false, l2_arr_dirty_addr);
                    }
                    if (l2_size != 0)
                        l2_cache(l2_tag_storage, l2_arr_valid, l2_arr_dirty, sb_valid, sb_top, sb_rec,
                                 sb_blocks, l2_pref_M, l2_pref_N, l2_counters, cache_block_size, l2_size,
                                 l2_assoc, l1_addr, 'r', l2_arr_lru, false, l2_arr_dirty_addr);

                    l1_tag_storage[c_set * c_assoc + j] = c_tag;
                    lru_policy(c_assoc, c_set, j, l1_arr_lru);
                    l1_arr_valid[c_set * c_assoc + j] = 1;
                    if (rw == 'w') {
                        l1_arr_dirty[c_set * c_assoc + j] = 'D';
                        l1_arr_dirty_addr[c_set * c_assoc + j] = l1_addr;
                    } else {
                        l1_arr_dirty[c_set * c_assoc + j] = ' ';
                        l1_arr_dirty_addr[c_set * c_assoc + j] = 0;
                    }
                    break;
                }
            }

            if (l1_pref_N > 0 && l1_pref_M > 0) {
                int sbp = sb_lru(l1_pref_N, sb_valid, sb_rec);
                if (sbp != -1) {
                    sb_miss(sbp, block_number, sb_valid, sb_top, sb_blocks, l1_pref_M, l1_pref_N, sb_rec);
                    for (uint32_t i = 0; i < l1_pref_M; ++i) {
                        uint32_t blk = sb_blocks[sbp * l1_pref_M + i];
                        uint32_t prefetch_addr = blk << l1_blockoffset_bits;
                        l1_counters.prefetch_req++;
                        if (l2_size != 0)
                            l2_cache(l2_tag_storage, l2_arr_valid, l2_arr_dirty, sb_valid, sb_top, sb_rec,
                                     sb_blocks, l2_pref_M, l2_pref_N, l2_counters, cache_block_size, l2_size,
                                     l2_assoc, prefetch_addr, 'r', l2_arr_lru, true, l2_arr_dirty_addr);
                    }
                }
            }
        }
        else
        {
            // miss + stream buffer hit
            if (rw == 'r') l1_counters.sb_read_hits++;
            else l1_counters.sb_write_hits++;

            for (int j = 0; j < c_assoc; ++j) {
                if (l1_arr_lru[c_set * c_assoc + j] == (c_assoc - 1)) {
                    if (l1_arr_dirty[c_set * c_assoc + j] == 'D') {
                        l1_counters.writeback_to_lower_mem++;
                        if (l2_size != 0)
                            l2_cache(l2_tag_storage, l2_arr_valid, l2_arr_dirty, sb_valid, sb_top, sb_rec,
                                     sb_blocks, l2_pref_M, l2_pref_N, l2_counters, cache_block_size, l2_size,
                                     l2_assoc, l1_arr_dirty_addr[c_set * c_assoc + j], 'w', l2_arr_lru, false, l2_arr_dirty_addr);
                    }
                    if (rw == 'w' && l2_size != 0)
                        l2_cache(l2_tag_storage, l2_arr_valid, l2_arr_dirty, sb_valid, sb_top, sb_rec,
                                 sb_blocks, l2_pref_M, l2_pref_N, l2_counters, cache_block_size, l2_size,
                                 l2_assoc, l1_addr, 'r', l2_arr_lru, false, l2_arr_dirty_addr);

                    l1_tag_storage[c_set * c_assoc + j] = c_tag;
                    lru_policy(c_assoc, c_set, j, l1_arr_lru);
                    l1_arr_valid[c_set * c_assoc + j] = 1;
                    if (rw == 'w') {
                        l1_arr_dirty[c_set * c_assoc + j] = 'D';
                        l1_arr_dirty_addr[c_set * c_assoc + j] = l1_addr;
                    } else {
                        l1_arr_dirty[c_set * c_assoc + j] = ' ';
                        l1_arr_dirty_addr[c_set * c_assoc + j] = 0;
                    }
                    break;
                }
            }

            int pos = sb_pos_finder(sb_hit_p_index, block_number, l1_pref_N, l1_pref_M, sb_top, sb_blocks);
            if (pos >= 0) {
                sb_hit_continue(sb_hit_p_index, pos, l1_pref_M, l1_pref_N, sb_top, sb_blocks, sb_rec);
                uint32_t new_count = pos + 1;
                uint32_t tail_start = (sb_top[sb_hit_p_index] + l1_pref_M - new_count) % l1_pref_M;
                for (uint32_t i = 0; i < new_count; i++) {
                    uint32_t real_index = (tail_start + i) % l1_pref_M;
                    uint32_t blk = sb_blocks[sb_hit_p_index * l1_pref_M + real_index];
                    uint32_t prefetch_addr = blk << l1_blockoffset_bits;
                    l1_counters.prefetch_req++;
                    if (l2_size != 0)
                        l2_cache(l2_tag_storage, l2_arr_valid, l2_arr_dirty, sb_valid, sb_top, sb_rec,
                                 sb_blocks, l2_pref_M, l2_pref_N, l2_counters, cache_block_size, l2_size,
                                 l2_assoc, prefetch_addr, 'r', l2_arr_lru, true, l2_arr_dirty_addr);
                }
            }
        }
    }
}


#endif


Writing cache_sim_flat.h


In [30]:
%%writefile cache_sim_flat.cc
#include <stdio.h>
#include <stdlib.h>
#include <inttypes.h>
#include <cmath>
#include <tuple>
#include <cstdint>
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <chrono>
#include "cache_sim_flat.h"
using namespace std;
#include <cstring>
int main(int argc, char *argv[])
{
	FILE *fp;
	char *trace_file;
	char rw;
	uint32_t addr;
	if (argc != 2)
	{
	    printf("Error: Expected 1 command-line argument (trace file) but was provided %d.\n", (argc - 1));
	    exit(EXIT_FAILURE);
	}
	trace_file = argv[1];

	uint32_t fixed_blocksize = 32;
	uint32_t fixed_l1_size = 1024;

	const int num_configs = 6;
	fp = fopen(trace_file, "r");
	if (fp == (FILE *)NULL) {
	    printf("Error: Unable to open file %s\n", trace_file);
	    exit(EXIT_FAILURE);
	}
	int total_no_of_ref = 0;
	{
	    char tmp_rw;
	    uint32_t tmp_addr;
	    while (fscanf(fp, "%c %x\n", &tmp_rw, &tmp_addr) == 2)
	    {
	        total_no_of_ref++;
	    }
	}
	rewind(fp);
	char* trace_rw = new char[total_no_of_ref];
	uint32_t* trace_addr = new uint32_t[total_no_of_ref];
	int idx = 0;
	while (fscanf(fp, "%c %x\n", &rw, &addr) == 2)
	{
	    trace_rw[idx] = rw;
	    trace_addr[idx] = addr;
	    idx++;
	}
	printf("===== Simulator configuration =====\n");
	printf("BLOCKSIZE:  %u\n", fixed_blocksize);
	printf("L1_SIZE:    %u\n", fixed_l1_size);
	printf("trace_file: %s\n", trace_file);
	printf("\n");
	int assoc_values[6] = {1, 2, 4, 8, 16, 32};
	auto t0 = std::chrono::high_resolution_clock::now();
	for (int cfg = 0; cfg < 6; ++cfg)
	{
	    uint32_t this_assoc = assoc_values[cfg];
	    int l1_no_of_sets = fixed_l1_size / (fixed_blocksize * this_assoc);
	    int* l1_arr_valid = new int[l1_no_of_sets * this_assoc];
	    memset(l1_arr_valid, 0, l1_no_of_sets * this_assoc * sizeof(int));
	    char* l1_arr_dirty = new char[l1_no_of_sets * this_assoc];
	    memset(l1_arr_dirty, ' ', l1_no_of_sets * this_assoc * sizeof(char));
	    int* l1_tag_storage = new int[l1_no_of_sets * this_assoc];
	    memset(l1_tag_storage, 0, l1_no_of_sets * this_assoc * sizeof(int));
	    uint32_t* l1_arr_dirty_addr = new uint32_t[l1_no_of_sets * this_assoc];
	    memset(l1_arr_dirty_addr, 0, l1_no_of_sets * this_assoc * sizeof(uint32_t));
	    int* l1_arr_lru = new int[l1_no_of_sets * this_assoc];
	    for (int i = 0; i < l1_no_of_sets; ++i)
	        for (uint32_t j = 0; j < this_assoc; ++j)
	            l1_arr_lru[i * this_assoc + j] = j;
	    cache_params l1_counters;
	    cache_params l2_counters;
	    for (int i = 0; i < total_no_of_ref; ++i)
	    {
	        l1_cache(l1_tag_storage, l1_arr_dirty, l1_arr_valid, l1_arr_dirty_addr, l1_arr_lru,
	                 nullptr, nullptr, nullptr, nullptr, nullptr,
	                 l1_counters, l2_counters, fixed_blocksize, fixed_l1_size, this_assoc,
	                 trace_addr[i], trace_rw[i], 0, 0,
	                 nullptr, nullptr, nullptr, nullptr, 0, 0, 0, 0);
	    }
	    double miss_rate = static_cast<double>(l1_counters.write_miss + l1_counters.read_miss)
	        / (l1_counters.read_hit + l1_counters.write_hit + l1_counters.read_miss + l1_counters.write_miss);
	    printf("L1_ASSOC = %2u  ->  L1 miss rate = %.4f\n", this_assoc, miss_rate);
	    delete[] l1_arr_valid;
	    delete[] l1_arr_dirty;
	    delete[] l1_tag_storage;
	    delete[] l1_arr_dirty_addr;
	    delete[] l1_arr_lru;
	}
	auto t1 = std::chrono::high_resolution_clock::now();
	double cpu_ms = std::chrono::duration<double, std::milli>(t1 - t0).count();
	printf("\nCPU sweep time: %.3f ms\n", cpu_ms);
	delete[] trace_rw;
	delete[] trace_addr;
	return 0;
}

Overwriting cache_sim_flat.cc


In [31]:
!g++ -std=c++17 -O2 -o cache_sim_flat cache_sim_flat.cc -I. && echo "Compiled successfully"

Compiled successfully


In [32]:
!time ./cache_sim_flat gcc_trace.txt

===== Simulator configuration =====
BLOCKSIZE:  32
L1_SIZE:    1024
trace_file: gcc_trace.txt

L1_ASSOC =  1  ->  L1 miss rate = 0.1935
L1_ASSOC =  2  ->  L1 miss rate = 0.1560
L1_ASSOC =  4  ->  L1 miss rate = 0.1427
L1_ASSOC =  8  ->  L1 miss rate = 0.1363
L1_ASSOC = 16  ->  L1 miss rate = 0.1362
L1_ASSOC = 32  ->  L1 miss rate = 0.1370

CPU sweep time: 50.233 ms

real	0m0.086s
user	0m0.083s
sys	0m0.002s


This is the GPU side with changing only associativity values (so there are 6 threads)

In [2]:
%%writefile cache_sim_gpu.h
#ifndef SIM_CACHE_GPU_H
#define SIM_CACHE_GPU_H
#include <cmath>
#include <tuple>
#include <utility>
#include <cstdint>
#include <climits>

typedef
struct {
   uint32_t BLOCKSIZE;
   uint32_t L1_SIZE;
   uint32_t L1_ASSOC;
   uint32_t L2_SIZE;
   uint32_t L2_ASSOC;
  uint32_t PREF_N;
  uint32_t PREF_M;
} cache_params_t;


//structure for all the cache parameters
struct cache_params
{
    int read_hit = 0;
    int write_hit = 0;
    int read_miss = 0;
    int write_miss = 0;
    int writeback_to_lower_mem = 0;

    int prefetch_reads_from_l1 = 0;
    int prefetch_misses_from_l1 = 0;

    int prefetch_req = 0;  //prefetches issued

    //hits on stream buffer
    int sb_read_hits = 0;
    int sb_write_hits = 0;
};



	//for implemetation of lru policy wherein counters are maintained indicating the recency
	//0: most recently used
	__host__ __device__ void lru_policy(int l_assoc, int l_index_set, int l_index_tag, int* l_arr_lru)
  {
    for (int q = 0; q < l_assoc; ++q) {
        if (l_arr_lru[l_index_set * l_assoc + q] < l_arr_lru[l_index_set * l_assoc + l_index_tag])
            l_arr_lru[l_index_set * l_assoc + q]++;
    }
    l_arr_lru[l_index_set * l_assoc + l_index_tag] = 0;
  }


  //getting the number of sets and number of bits required for set and blockoffset
	__host__ __device__ std::tuple<int, int, int> set_tag_bits_cal(int block_size, int cache_size, int assoc)
	{
	    int no_of_sets = cache_size / (block_size * assoc);
	    int set_bits = (int)std::round(std::log2(no_of_sets));
      int blockoffset_bits = (int)std::round(std::log2(block_size));
	    return {set_bits, blockoffset_bits, no_of_sets};
	}
  //parsing out the set and tag fields out of the 32 bits address
	__host__ __device__ std::pair<uint32_t, uint32_t> parseaddress(uint32_t address, int no_set_bits, int no_blockoffset_bits)
 	{
	    uint32_t index_mask = (1u << no_set_bits) - 1;
	    uint32_t set = (address >> no_blockoffset_bits) & index_mask;
	    uint32_t tag = address >> (no_blockoffset_bits + no_set_bits);
	    return {tag, set};
	}

  __host__ __device__ void sb_mark_mru(int p_index, int* sb_rec, uint32_t pref_N)
  {
    if (pref_N == 0)
    return;
    for (size_t i = 0; i < pref_N; ++i) {
        if ((int)i != p_index)
            sb_rec[i] += 1;
    }
    sb_rec[p_index] = 0;
  }
  __host__ __device__ int sb_hit(uint32_t pref_N, uint32_t pref_M, bool* sb_valid, int* sb_top, int* sb_rec, uint32_t* sb_blocks, uint32_t block_number)
  {
    if (pref_N == 0) return -1;
    int selected = -1;
    int best_rec = INT32_MAX;
    for (size_t i = 0; i < pref_N; ++i) {
        if (!sb_valid[i]) continue;
        for (uint32_t j = 0; j < pref_M; ++j) {
            uint32_t real_index = (sb_top[i] + j) % pref_M;
            if (sb_blocks[i*pref_M + real_index] == block_number) {
                if (sb_rec[i] < best_rec) {
                    selected = (int)i;
                    best_rec = sb_rec[i];
                }
                break;
            }
        }
    }
    return selected;
  }

  __host__ __device__ int sb_pos_finder(int buf_p_index, uint32_t block_number, uint32_t pref_N, uint32_t pref_M, int* sb_top, uint32_t* sb_blocks)
  {
    if (buf_p_index < 0 || (uint32_t)buf_p_index >= pref_N) return -1;
    for (uint32_t j = 0; j < pref_M; ++j) {
        uint32_t real_index = (sb_top[buf_p_index] + j) % pref_M;
        if (sb_blocks[buf_p_index*pref_M+real_index] == block_number)
            return (int)j;
    }
    return -1;
  }

  __host__ __device__ void sb_miss(int buf_p_index, uint32_t start_block, bool* sb_valid, int* sb_top, uint32_t* sb_blocks, uint32_t pref_M, uint32_t pref_N, int* sb_rec)
  {
    if (pref_N == 0) return;
    sb_valid[buf_p_index] = true;
    sb_top[buf_p_index] = 0;
    uint32_t next = start_block + 1;
    for (uint32_t i = 0; i < pref_M; ++i)
        sb_blocks[buf_p_index*pref_M + i] = next + i;
    sb_mark_mru(buf_p_index, sb_rec, pref_N);
  }

  __host__ __device__ void sb_hit_continue(int buf_p_index, int pos, uint32_t pref_M, uint32_t pref_N,
                     int* sb_top, uint32_t* sb_blocks, int* sb_rec)
  {
    if (pref_N == 0) return;
    int count =0;
    uint32_t* remaining = new uint32_t[pref_M];
    for (uint32_t i = pos + 1; i < pref_M; ++i) {
        uint32_t real_index = (sb_top[buf_p_index] + i) % pref_M;
        remaining[count++] = sb_blocks[buf_p_index * pref_M + real_index];
    }
    uint32_t last_elem_index = (sb_top[buf_p_index]+ pref_M - 1) % pref_M;
    uint32_t next_prefetch = sb_blocks[buf_p_index * pref_M + last_elem_index] + 1;
    uint32_t* newblocks = new uint32_t[pref_M];
    int nb_count = 0;
    for (int k = 0; k < count; ++k)
      newblocks[nb_count++] = remaining[k];
    while (nb_count < (int)pref_M)
      newblocks[nb_count++] = next_prefetch++;
    for (uint32_t i = 0; i < pref_M; ++i)
      sb_blocks[buf_p_index * pref_M + i] = newblocks[i];
    sb_top[buf_p_index] = 0;
    sb_mark_mru(buf_p_index, sb_rec, pref_N);
    delete[] remaining;
    delete[] newblocks;
  }

  __host__ __device__ int sb_lru(uint32_t pref_N, bool* sb_valid, int* sb_rec)
  {
    if (pref_N == 0) return -1;
    for (size_t i = 0; i < pref_N; ++i)
        if (!sb_valid[i]) return (int)i;
    int best_p_index = 0;
    int best_rec = sb_rec[0];
    for (size_t i = 1; i < pref_N; ++i) {
        if (sb_rec[i] > best_rec) {
            best_rec = sb_rec[i];
            best_p_index = (int)i;
        }
    }
    return best_p_index;
  }
  __host__ __device__ void l2_cache(int* l2_tag_storage, int* l2_arr_valid, char* l2_arr_dirty, bool* sb_valid, int* sb_top, int* sb_rec, uint32_t* sb_blocks, uint32_t pref_M, uint32_t pref_N, cache_params& l2_counters, int l2_block_size, int l2_size, int l2_assoc, uint32_t l2_addr, char l2_rw, int* l2_arr_lru, bool is_prefetch, uint32_t* l2_arr_dirty_addr)
	{
	    auto [l2_set_bits, l2_blockoffset_bits, l2_no_of_sets] = set_tag_bits_cal(l2_block_size, l2_size, l2_assoc);
	    auto [l2_tag, l2_set] = parseaddress(l2_addr, l2_set_bits, l2_blockoffset_bits);
      if (l2_rw == 'r') {
        if (is_prefetch)
        l2_counters.prefetch_reads_from_l1++;
      }
      bool l2_search_tag = false;
      int index_tag = -1;
      for (int i = 0; i < l2_assoc; ++i) {
      if (l2_arr_valid[l2_set*l2_assoc + i] == 1 && l2_tag == l2_tag_storage[l2_set *l2_assoc + i]) {
        l2_search_tag = true;
        index_tag = i;
        break;
      }
      }

      uint32_t block_number = (l2_addr >> l2_blockoffset_bits);
      int sb_hit_p_index = -1;
      if (pref_N > 0) {
      sb_hit_p_index = sb_hit(pref_N, pref_M, sb_valid, sb_top, sb_rec,sb_blocks, block_number);
      }

      if (l2_search_tag)
      {
          lru_policy(l2_assoc, l2_set, index_tag, l2_arr_lru);
          if (l2_rw == 'r') {
          if (!is_prefetch)
            l2_counters.read_hit++;
          } else {
          l2_counters.write_hit++;
          l2_arr_dirty[l2_set * l2_assoc + index_tag] = 'D';
          }
          if (sb_hit_p_index != -1) {
            int pos = sb_pos_finder(sb_hit_p_index, block_number, pref_N, pref_M, sb_top, sb_blocks);
            if (pos >= 0) {
            sb_hit_continue(sb_hit_p_index, pos, pref_M, pref_N,
              sb_top, sb_blocks, sb_rec );
              uint32_t new_count = pos + 1;
              for (uint32_t i = 0; i < new_count; i++)
                l2_counters.prefetch_req++;
            }
          }
      }
      else
{
    if (sb_hit_p_index == -1)
    {
        if (is_prefetch)
            l2_counters.prefetch_misses_from_l1++;
        else {
            if (l2_rw == 'r') l2_counters.read_miss++;
            else l2_counters.write_miss++;
        }
    }
    else
    {
        if (!is_prefetch) {
            if (l2_rw == 'r') l2_counters.sb_read_hits++;
            else l2_counters.sb_write_hits++;
        }
    }
    for (int j = 0; j < l2_assoc; ++j) {
        if (l2_arr_lru[l2_set * l2_assoc + j] == (l2_assoc - 1)) {
            if (l2_arr_dirty[l2_set * l2_assoc + j] == 'D')
                l2_counters.writeback_to_lower_mem++;
            l2_tag_storage[l2_set * l2_assoc + j] = l2_tag;
            l2_arr_dirty[l2_set * l2_assoc + j] = (l2_rw == 'w') ? 'D' : ' ';
            l2_arr_valid[l2_set * l2_assoc + j] = 1;
            lru_policy(l2_assoc, l2_set, j, l2_arr_lru);
            break;
        }
    }
    if (sb_hit_p_index == -1) {
        if (pref_N > 0 && pref_M > 0 && !is_prefetch) {
            int sbp = sb_lru(pref_N, sb_valid, sb_rec);
            if (sbp != -1) {
                sb_miss(sbp,block_number, sb_valid, sb_top, sb_blocks, pref_M, pref_N, sb_rec);
                for (uint32_t i = 0; i < pref_M; ++i)
                    l2_counters.prefetch_req++;
            }
        }
        } else {
            int pos = sb_pos_finder(sb_hit_p_index, block_number, pref_N, pref_M, sb_top, sb_blocks);
            if (pos >= 0) {
            sb_hit_continue(sb_hit_p_index, pos, pref_M, pref_N, sb_top, sb_blocks, sb_rec);
            uint32_t new_count = pos + 1;
            for (uint32_t i = 0; i < new_count; i++)
                l2_counters.prefetch_req++;
        }
    }
  }
  }

  __host__ __device__ void l1_cache(int* l1_tag_storage, char* l1_arr_dirty, int* l1_arr_valid, uint32_t* l1_arr_dirty_addr, int* l1_arr_lru,
              int* l2_tag_storage, char* l2_arr_dirty, int* l2_arr_valid, uint32_t* l2_arr_dirty_addr,
              int* l2_arr_lru, cache_params& l1_counters, cache_params& l2_counters,
              int cache_block_size, int cache_size, int c_assoc, uint32_t l1_addr, char rw,
              int l2_size, int l2_assoc, bool* sb_valid, int* sb_top, int* sb_rec, uint32_t* sb_blocks,
              uint32_t l1_pref_N, uint32_t l1_pref_M, uint32_t l2_pref_N, uint32_t l2_pref_M)
{
    auto [l1_set_bits, l1_blockoffset_bits, l1_no_of_sets] = set_tag_bits_cal(cache_block_size, cache_size, c_assoc);
    auto [c_tag, c_set] = parseaddress(l1_addr, l1_set_bits, l1_blockoffset_bits);

    bool l1_search_tag = false;
    int index_tag = -1;
    for (int i = 0; i < c_assoc; ++i) {
        if (l1_arr_valid[c_set * c_assoc + i] == 1 && c_tag == l1_tag_storage[c_set * c_assoc + i]) {
            l1_search_tag = true;
            index_tag = i;
            break;
        }
    }

    uint32_t block_number = (l1_addr >> l1_blockoffset_bits);
    int sb_hit_p_index = -1;
    if (l1_pref_N > 0) {
        sb_hit_p_index = sb_hit(l1_pref_N, l1_pref_M, sb_valid, sb_top, sb_rec, sb_blocks, block_number);
    }

    if (l1_search_tag)
    {
        // ---- cache hit ----
        lru_policy(c_assoc, c_set, index_tag, l1_arr_lru);
        if (rw == 'r') {
            l1_counters.read_hit++;
        } else {
            l1_counters.write_hit++;
            l1_arr_dirty[c_set * c_assoc + index_tag] = 'D';
            l1_arr_dirty_addr[c_set * c_assoc + index_tag] = l1_addr;
        }

        // hit + stream buffer hit
        if (sb_hit_p_index != -1) {
            int pos = sb_pos_finder(sb_hit_p_index, block_number, l1_pref_N, l1_pref_M, sb_top, sb_blocks);
            if (pos >= 0) {
                sb_hit_continue(sb_hit_p_index, pos, l1_pref_M, l1_pref_N, sb_top, sb_blocks, sb_rec);
                uint32_t new_count = pos + 1;
                uint32_t tail_start = (sb_top[sb_hit_p_index] + l1_pref_M - new_count) % l1_pref_M;
                for (uint32_t i = 0; i < new_count; i++) {
                    uint32_t real_index = (tail_start + i) % l1_pref_M;
                    uint32_t blk = sb_blocks[sb_hit_p_index * l1_pref_M + real_index];
                    uint32_t prefetch_addr = blk << l1_blockoffset_bits;
                    l1_counters.prefetch_req++;
                    if (l2_size != 0)
                        l2_cache(l2_tag_storage, l2_arr_valid, l2_arr_dirty, sb_valid, sb_top, sb_rec,
                                 sb_blocks, l2_pref_M, l2_pref_N, l2_counters, cache_block_size, l2_size,
                                 l2_assoc, prefetch_addr, 'r', l2_arr_lru, true, l2_arr_dirty_addr);
                }
            }
        }
    }
    else
    {
        // ---- cache miss ----
        if (sb_hit_p_index == -1)
        {
            // miss + stream buffer miss
            if (rw == 'r') l1_counters.read_miss++;
            else l1_counters.write_miss++;

            for (int j = 0; j < c_assoc; ++j) {
                if (l1_arr_lru[c_set * c_assoc + j] == (c_assoc - 1)) {
                    if (l1_arr_dirty[c_set * c_assoc + j] == 'D') {
                        l1_counters.writeback_to_lower_mem++;
                        if (l2_size != 0)
                            l2_cache(l2_tag_storage, l2_arr_valid, l2_arr_dirty, sb_valid, sb_top, sb_rec,
                                     sb_blocks, l2_pref_M, l2_pref_N, l2_counters, cache_block_size, l2_size,
                                     l2_assoc, l1_arr_dirty_addr[c_set * c_assoc + j], 'w', l2_arr_lru, false, l2_arr_dirty_addr);
                    }
                    if (l2_size != 0)
                        l2_cache(l2_tag_storage, l2_arr_valid, l2_arr_dirty, sb_valid, sb_top, sb_rec,
                                 sb_blocks, l2_pref_M, l2_pref_N, l2_counters, cache_block_size, l2_size,
                                 l2_assoc, l1_addr, 'r', l2_arr_lru, false, l2_arr_dirty_addr);

                    l1_tag_storage[c_set * c_assoc + j] = c_tag;
                    lru_policy(c_assoc, c_set, j, l1_arr_lru);
                    l1_arr_valid[c_set * c_assoc + j] = 1;
                    if (rw == 'w') {
                        l1_arr_dirty[c_set * c_assoc + j] = 'D';
                        l1_arr_dirty_addr[c_set * c_assoc + j] = l1_addr;
                    } else {
                        l1_arr_dirty[c_set * c_assoc + j] = ' ';
                        l1_arr_dirty_addr[c_set * c_assoc + j] = 0;
                    }
                    break;
                }
            }

            if (l1_pref_N > 0 && l1_pref_M > 0) {
                int sbp = sb_lru(l1_pref_N, sb_valid, sb_rec);
                if (sbp != -1) {
                    sb_miss(sbp, block_number, sb_valid, sb_top, sb_blocks, l1_pref_M, l1_pref_N, sb_rec);
                    for (uint32_t i = 0; i < l1_pref_M; ++i) {
                        uint32_t blk = sb_blocks[sbp * l1_pref_M + i];
                        uint32_t prefetch_addr = blk << l1_blockoffset_bits;
                        l1_counters.prefetch_req++;
                        if (l2_size != 0)
                            l2_cache(l2_tag_storage, l2_arr_valid, l2_arr_dirty, sb_valid, sb_top, sb_rec,
                                     sb_blocks, l2_pref_M, l2_pref_N, l2_counters, cache_block_size, l2_size,
                                     l2_assoc, prefetch_addr, 'r', l2_arr_lru, true, l2_arr_dirty_addr);
                    }
                }
            }
        }
        else
        {
            // miss + stream buffer hit
            if (rw == 'r') l1_counters.sb_read_hits++;
            else l1_counters.sb_write_hits++;

            for (int j = 0; j < c_assoc; ++j) {
                if (l1_arr_lru[c_set * c_assoc + j] == (c_assoc - 1)) {
                    if (l1_arr_dirty[c_set * c_assoc + j] == 'D') {
                        l1_counters.writeback_to_lower_mem++;
                        if (l2_size != 0)
                            l2_cache(l2_tag_storage, l2_arr_valid, l2_arr_dirty, sb_valid, sb_top, sb_rec,
                                     sb_blocks, l2_pref_M, l2_pref_N, l2_counters, cache_block_size, l2_size,
                                     l2_assoc, l1_arr_dirty_addr[c_set * c_assoc + j], 'w', l2_arr_lru, false, l2_arr_dirty_addr);
                    }
                    if (rw == 'w' && l2_size != 0)
                        l2_cache(l2_tag_storage, l2_arr_valid, l2_arr_dirty, sb_valid, sb_top, sb_rec,
                                 sb_blocks, l2_pref_M, l2_pref_N, l2_counters, cache_block_size, l2_size,
                                 l2_assoc, l1_addr, 'r', l2_arr_lru, false, l2_arr_dirty_addr);

                    l1_tag_storage[c_set * c_assoc + j] = c_tag;
                    lru_policy(c_assoc, c_set, j, l1_arr_lru);
                    l1_arr_valid[c_set * c_assoc + j] = 1;
                    if (rw == 'w') {
                        l1_arr_dirty[c_set * c_assoc + j] = 'D';
                        l1_arr_dirty_addr[c_set * c_assoc + j] = l1_addr;
                    } else {
                        l1_arr_dirty[c_set * c_assoc + j] = ' ';
                        l1_arr_dirty_addr[c_set * c_assoc + j] = 0;
                    }
                    break;
                }
            }

            int pos = sb_pos_finder(sb_hit_p_index, block_number, l1_pref_N, l1_pref_M, sb_top, sb_blocks);
            if (pos >= 0) {
                sb_hit_continue(sb_hit_p_index, pos, l1_pref_M, l1_pref_N, sb_top, sb_blocks, sb_rec);
                uint32_t new_count = pos + 1;
                uint32_t tail_start = (sb_top[sb_hit_p_index] + l1_pref_M - new_count) % l1_pref_M;
                for (uint32_t i = 0; i < new_count; i++) {
                    uint32_t real_index = (tail_start + i) % l1_pref_M;
                    uint32_t blk = sb_blocks[sb_hit_p_index * l1_pref_M + real_index];
                    uint32_t prefetch_addr = blk << l1_blockoffset_bits;
                    l1_counters.prefetch_req++;
                    if (l2_size != 0)
                        l2_cache(l2_tag_storage, l2_arr_valid, l2_arr_dirty, sb_valid, sb_top, sb_rec,
                                 sb_blocks, l2_pref_M, l2_pref_N, l2_counters, cache_block_size, l2_size,
                                 l2_assoc, prefetch_addr, 'r', l2_arr_lru, true, l2_arr_dirty_addr);
                }
            }
        }
    }
}


#endif


Writing cache_sim_gpu.h


In [23]:
%%writefile cache_sim_gpu.cu
#include <stdio.h>
#include <stdlib.h>
#include <inttypes.h>
#include <cmath>
#include <tuple>
#include <cstdint>
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include "cache_sim_gpu.h"
using namespace std;
#include <cstring>

__global__ void sweep_kernel(int num_configs, int max_sets, int max_assoc,
                              uint32_t* assoc_values_arr,
                              int* l1_arr_valid, int* l1_tag_storage, char* l1_arr_dirty,
                              uint32_t* l1_arr_dirty_addr, int* l1_arr_lru,
                              cache_params* counters,
                              uint32_t blocksize, uint32_t l1_size,
                              char* trace_rw, uint32_t* trace_addr, int total_no_of_ref)
  {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid >= num_configs) return;

    uint32_t this_assoc = assoc_values_arr[tid];
    int this_no_of_sets = l1_size / (blocksize * this_assoc);
    size_t base = (size_t)tid * max_sets * max_assoc;

    int* my_valid       = l1_arr_valid + base;
    int* my_tags        = l1_tag_storage + base;
    char* my_dirty       = l1_arr_dirty + base;
    uint32_t* my_dirty_addr = l1_arr_dirty_addr + base;
    int* my_lru         = l1_arr_lru + base;

    for (int s = 0; s < this_no_of_sets; ++s)
        for (uint32_t w = 0; w < this_assoc; ++w)
            my_lru[s * this_assoc + w] = w;

    cache_params my_counters;
    cache_params dummy_l2_counters;

    for (int i = 0; i < total_no_of_ref; ++i)
    {
        l1_cache(my_tags, my_dirty, my_valid, my_dirty_addr, my_lru,
                 nullptr, nullptr, nullptr, nullptr, nullptr,
                 my_counters, dummy_l2_counters,
                 blocksize, l1_size, this_assoc,
                 trace_addr[i], trace_rw[i], 0, 0,
                 nullptr, nullptr, nullptr, nullptr, 0, 0, 0, 0);
    }

    counters[tid] = my_counters;
  }




int main(int argc, char *argv[])
{
	FILE *fp;
	char *trace_file;
	char rw;
	uint32_t addr;

	if (argc != 2)
	{
	    printf("Error: Expected 1 command-line argument (trace file) but was provided %d.\n", (argc - 1));
	    exit(EXIT_FAILURE);
	}

	trace_file = argv[1];

	uint32_t fixed_blocksize = 32;
	uint32_t fixed_l1_size = 1024;

	const int num_configs = 6;
	int assoc_values[6] = {1, 2, 4, 8, 16, 32};
	int max_assoc = 32;
	int max_sets  = fixed_l1_size / (fixed_blocksize * 1);

	fp = fopen(trace_file, "r");
	if (fp == (FILE *)NULL) {
	    printf("Error: Unable to open file %s\n", trace_file);
	    exit(EXIT_FAILURE);
	}
  int total_no_of_ref = 0;
  {
    char tmp_rw;
    uint32_t tmp_addr;
	  while (fscanf(fp, "%c %x\n", &tmp_rw, &tmp_addr) == 2)
	  {
	    total_no_of_ref++;
	  }
  }
  rewind(fp);
  char* trace_rw = new char[total_no_of_ref];
  uint32_t* trace_addr = new uint32_t[total_no_of_ref];
  int idx = 0;
  while (fscanf(fp, "%c %x\n", &rw, &addr) == 2)
  {
    trace_rw[idx] = rw;
    trace_addr[idx] = addr;
    idx++;
  }


	printf("===== Simulator configuration =====\n");
	printf("BLOCKSIZE:  %u\n", fixed_blocksize);
	printf("L1_SIZE:    %u\n", fixed_l1_size);
	printf("trace_file: %s\n", trace_file);
	printf("\n");

  int *g_l1_arr_valid, *g_l1_tag_storage, *g_l1_arr_lru;
  char *g_l1_arr_dirty;
  uint32_t *g_l1_arr_dirty_addr;
  cache_params *g_counters;
  uint32_t *g_assoc_values;
  char *g_trace_rw;
  uint32_t *g_trace_addr;

  size_t per_thread_size = max_sets * max_assoc;

  cudaMallocManaged(&g_l1_arr_valid,      num_configs * per_thread_size * sizeof(int));
  cudaMallocManaged(&g_l1_tag_storage,    num_configs * per_thread_size * sizeof(int));
  cudaMallocManaged(&g_l1_arr_dirty,      num_configs * per_thread_size * sizeof(char));
  cudaMallocManaged(&g_l1_arr_dirty_addr, num_configs * per_thread_size * sizeof(uint32_t));
  cudaMallocManaged(&g_l1_arr_lru,        num_configs * per_thread_size * sizeof(int));
  cudaMallocManaged(&g_counters,          num_configs * sizeof(cache_params));
  cudaMallocManaged(&g_assoc_values,      num_configs * sizeof(uint32_t));
  cudaMallocManaged(&g_trace_rw,          total_no_of_ref * sizeof(char));
  cudaMallocManaged(&g_trace_addr,        total_no_of_ref * sizeof(uint32_t));

  for (int i = 0; i < num_configs; ++i) g_assoc_values[i] = assoc_values[i];
  for (int i = 0; i < total_no_of_ref; ++i) { g_trace_rw[i] = trace_rw[i]; g_trace_addr[i] = trace_addr[i]; }

  memset(g_l1_arr_valid, 0, num_configs * per_thread_size * sizeof(int));
  memset(g_l1_arr_dirty, ' ', num_configs * per_thread_size * sizeof(char));
  memset(g_l1_arr_dirty_addr, 0, num_configs * per_thread_size * sizeof(uint32_t));


  cudaEvent_t start, stop;
  cudaEventCreate(&start);
  cudaEventCreate(&stop);

  cudaEventRecord(start);
  int threadsPerBlock = 32;
  int numBlocks = (num_configs + threadsPerBlock - 1) / threadsPerBlock;
  sweep_kernel<<<numBlocks, threadsPerBlock>>>(num_configs, max_sets, max_assoc,
                                  g_assoc_values,
                                  g_l1_arr_valid, g_l1_tag_storage, g_l1_arr_dirty,
                                  g_l1_arr_dirty_addr, g_l1_arr_lru,
                                  g_counters,
                                  fixed_blocksize, fixed_l1_size,
                                  g_trace_rw, g_trace_addr, total_no_of_ref);
  cudaEventRecord(stop);

  cudaEventSynchronize(stop);   // waits for the kernel to finish, replaces cudaDeviceSynchronize() here
  float kernel_ms = 0;
  cudaEventElapsedTime(&kernel_ms, start, stop);
  printf("Kernel execution time: %.3f ms\n", kernel_ms);

  cudaEventDestroy(start);
  cudaEventDestroy(stop);
  for (int i = 0; i < num_configs; ++i) {
    cache_params &c = g_counters[i];
    double miss_rate = static_cast<double>(c.write_miss + c.read_miss)
        / (c.read_hit + c.write_hit + c.read_miss + c.write_miss);
    printf("L1_ASSOC = %2u  ->  L1 miss rate = %.4f\n", g_assoc_values[i], miss_rate);
  }

	delete[] trace_rw;
	delete[] trace_addr;
  cudaFree(g_l1_arr_valid);
  cudaFree(g_l1_tag_storage);
  cudaFree(g_l1_arr_dirty);
  cudaFree(g_l1_arr_dirty_addr);
  cudaFree(g_l1_arr_lru);
  cudaFree(g_counters);
  cudaFree(g_assoc_values);
  cudaFree(g_trace_rw);
  cudaFree(g_trace_addr);

	return 0;
}


Overwriting cache_sim_gpu.cu


In [24]:
!nvcc --expt-relaxed-constexpr cache_sim_gpu.cu -o cache_sim_gpu_cuda

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [25]:
!./cache_sim_gpu_cuda gcc_trace.txt

===== Simulator configuration =====
BLOCKSIZE:  32
L1_SIZE:    1024
trace_file: gcc_trace.txt

Kernel execution time: 744.766 ms
L1_ASSOC =  1  ->  L1 miss rate = 0.1935
L1_ASSOC =  2  ->  L1 miss rate = 0.1560
L1_ASSOC =  4  ->  L1 miss rate = 0.1427
L1_ASSOC =  8  ->  L1 miss rate = 0.1363
L1_ASSOC = 16  ->  L1 miss rate = 0.1362
L1_ASSOC = 32  ->  L1 miss rate = 0.1370


It was observed that the CPU time was very less as compared to the GPU kernel time and so I wanted to increase the threads from 6 to 48 by changing the blocksize as well


CPU code with 48 threads (associativity and cache size changes)

In [33]:
%%writefile cache_sim_flat.cc
#include <stdio.h>
#include <stdlib.h>
#include <inttypes.h>
#include <cmath>
#include <tuple>
#include <cstdint>
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <chrono>
#include "cache_sim_flat.h"
using namespace std;
#include <cstring>

int main(int argc, char *argv[])
{
	FILE *fp;
	char *trace_file;
	char rw;
	uint32_t addr;

	if (argc != 2)
	{
	    printf("Error: Expected 1 command-line argument (trace file) but was provided %d.\n", (argc - 1));
	    exit(EXIT_FAILURE);
	}

	trace_file = argv[1];

	uint32_t fixed_blocksize = 32;

	fp = fopen(trace_file, "r");
	if (fp == (FILE *)NULL) {
	    printf("Error: Unable to open file %s\n", trace_file);
	    exit(EXIT_FAILURE);
	}

	int total_no_of_ref = 0;
	{
	    char tmp_rw;
	    uint32_t tmp_addr;
	    while (fscanf(fp, "%c %x\n", &tmp_rw, &tmp_addr) == 2)
	    {
	        total_no_of_ref++;
	    }
	}
	rewind(fp);
	char* trace_rw = new char[total_no_of_ref];
	uint32_t* trace_addr = new uint32_t[total_no_of_ref];
	int idx = 0;
	while (fscanf(fp, "%c %x\n", &rw, &addr) == 2)
	{
	    trace_rw[idx] = rw;
	    trace_addr[idx] = addr;
	    idx++;
	}

	printf("===== Simulator configuration =====\n");
	printf("BLOCKSIZE:  %u\n", fixed_blocksize);
	printf("trace_file: %s\n", trace_file);
	printf("\n");
	int assoc_values[6] = {1, 2, 4, 8, 16, 32};
	int size_values[8] = {256, 512, 1024, 2048, 4096, 8192, 16384, 32768};
	int num_assoc = 6;
	int num_size = 8;
	int num_configs = num_assoc * num_size;   // 48

	auto t0 = std::chrono::high_resolution_clock::now();

	for (int cfg = 0; cfg < num_configs; ++cfg)
	{
	    uint32_t this_assoc = assoc_values[cfg / num_size];
	    uint32_t this_size  = size_values[cfg % num_size];
	    if (this_size < fixed_blocksize * this_assoc) {
	        printf("L1_ASSOC = %2u  L1_SIZE = %6u  ->  skipped (invalid: fewer than 1 set)\n", this_assoc, this_size);
	        continue;
	    }
	    int l1_no_of_sets = this_size / (fixed_blocksize * this_assoc);

	    int* l1_arr_valid = new int[l1_no_of_sets * this_assoc];
	    memset(l1_arr_valid, 0, l1_no_of_sets * this_assoc * sizeof(int));

	    char* l1_arr_dirty = new char[l1_no_of_sets * this_assoc];
	    memset(l1_arr_dirty, ' ', l1_no_of_sets * this_assoc * sizeof(char));

	    int* l1_tag_storage = new int[l1_no_of_sets * this_assoc];
	    memset(l1_tag_storage, 0, l1_no_of_sets * this_assoc * sizeof(int));

	    uint32_t* l1_arr_dirty_addr = new uint32_t[l1_no_of_sets * this_assoc];
	    memset(l1_arr_dirty_addr, 0, l1_no_of_sets * this_assoc * sizeof(uint32_t));

	    int* l1_arr_lru = new int[l1_no_of_sets * this_assoc];
	    for (int i = 0; i < l1_no_of_sets; ++i)
	        for (uint32_t j = 0; j < this_assoc; ++j)
	            l1_arr_lru[i * this_assoc + j] = j;

	    cache_params l1_counters;
	    cache_params l2_counters;

	    for (int i = 0; i < total_no_of_ref; ++i)
	    {
	        l1_cache(l1_tag_storage, l1_arr_dirty, l1_arr_valid, l1_arr_dirty_addr, l1_arr_lru,
	                 nullptr, nullptr, nullptr, nullptr, nullptr,
	                 l1_counters, l2_counters, fixed_blocksize, this_size, this_assoc,
	                 trace_addr[i], trace_rw[i], 0, 0,
	                 nullptr, nullptr, nullptr, nullptr, 0, 0, 0, 0);
	    }

	    double miss_rate = static_cast<double>(l1_counters.write_miss + l1_counters.read_miss)
	        / (l1_counters.read_hit + l1_counters.write_hit + l1_counters.read_miss + l1_counters.write_miss);

	    printf("L1_ASSOC = %2u  L1_SIZE = %6u  ->  L1 miss rate = %.4f\n", this_assoc, this_size, miss_rate);

	    delete[] l1_arr_valid;
	    delete[] l1_arr_dirty;
	    delete[] l1_tag_storage;
	    delete[] l1_arr_dirty_addr;
	    delete[] l1_arr_lru;
	}

	auto t1 = std::chrono::high_resolution_clock::now();
	double cpu_ms = std::chrono::duration<double, std::milli>(t1 - t0).count();
	printf("\nCPU sweep time: %.3f ms\n", cpu_ms);

	delete[] trace_rw;
	delete[] trace_addr;

	return 0;
}

Overwriting cache_sim_flat.cc


In [35]:
!g++ -std=c++17 -O2 -o cache_sim_flat cache_sim_flat.cc -I. && echo "Compiled successfully"
!time ./cache_sim_flat gcc_trace.txt

Compiled successfully
===== Simulator configuration =====
BLOCKSIZE:  32
trace_file: gcc_trace.txt

L1_ASSOC =  1  L1_SIZE =    256  ->  L1 miss rate = 0.3336
L1_ASSOC =  1  L1_SIZE =    512  ->  L1 miss rate = 0.2653
L1_ASSOC =  1  L1_SIZE =   1024  ->  L1 miss rate = 0.1935
L1_ASSOC =  1  L1_SIZE =   2048  ->  L1 miss rate = 0.1477
L1_ASSOC =  1  L1_SIZE =   4096  ->  L1 miss rate = 0.1002
L1_ASSOC =  1  L1_SIZE =   8192  ->  L1 miss rate = 0.0670
L1_ASSOC =  1  L1_SIZE =  16384  ->  L1 miss rate = 0.0461
L1_ASSOC =  1  L1_SIZE =  32768  ->  L1 miss rate = 0.0377
L1_ASSOC =  2  L1_SIZE =    256  ->  L1 miss rate = 0.3053
L1_ASSOC =  2  L1_SIZE =    512  ->  L1 miss rate = 0.2204
L1_ASSOC =  2  L1_SIZE =   1024  ->  L1 miss rate = 0.1560
L1_ASSOC =  2  L1_SIZE =   2048  ->  L1 miss rate = 0.1071
L1_ASSOC =  2  L1_SIZE =   4096  ->  L1 miss rate = 0.0753
L1_ASSOC =  2  L1_SIZE =   8192  ->  L1 miss rate = 0.0473
L1_ASSOC =  2  L1_SIZE =  16384  ->  L1 miss rate = 0.0338
L1_ASSOC =  2  

GPU code for 48 threads (associativity and cache size changes)

In [36]:
%%writefile cache_sim_gpu.cu
#include <stdio.h>
#include <stdlib.h>
#include <inttypes.h>
#include <cmath>
#include <tuple>
#include <cstdint>
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include "cache_sim_gpu.h"
using namespace std;
#include <cstring>

__global__ void sweep_kernel(int num_configs, int num_size, int max_sets, int max_assoc,
                              uint32_t* assoc_values_arr, uint32_t* size_values_arr,
                              int* l1_arr_valid, int* l1_tag_storage, char* l1_arr_dirty,
                              uint32_t* l1_arr_dirty_addr, int* l1_arr_lru,
                              cache_params* counters,
                              uint32_t blocksize,
                              char* trace_rw, uint32_t* trace_addr, int total_no_of_ref)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid >= num_configs) return;

    uint32_t this_assoc = assoc_values_arr[tid / num_size];
    uint32_t this_size  = size_values_arr[tid % num_size];
    int this_no_of_sets = this_size / (blocksize * this_assoc);
    if (this_no_of_sets == 0) {
    cache_params empty_counters;
    counters[tid] = empty_counters;
    return;
    }
    size_t base = (size_t)tid * max_sets * max_assoc;

    int* my_valid       = l1_arr_valid + base;
    int* my_tags        = l1_tag_storage + base;
    char* my_dirty       = l1_arr_dirty + base;
    uint32_t* my_dirty_addr = l1_arr_dirty_addr + base;
    int* my_lru         = l1_arr_lru + base;

    for (int s = 0; s < this_no_of_sets; ++s)
        for (uint32_t w = 0; w < this_assoc; ++w)
            my_lru[s * this_assoc + w] = w;

    cache_params my_counters;
    cache_params dummy_l2_counters;

    for (int i = 0; i < total_no_of_ref; ++i)
    {
        l1_cache(my_tags, my_dirty, my_valid, my_dirty_addr, my_lru,
                 nullptr, nullptr, nullptr, nullptr, nullptr,
                 my_counters, dummy_l2_counters,
                 blocksize, this_size, this_assoc,
                 trace_addr[i], trace_rw[i], 0, 0,
                 nullptr, nullptr, nullptr, nullptr, 0, 0, 0, 0);
    }

    counters[tid] = my_counters;
}
int main(int argc, char *argv[])
{
	FILE *fp;
	char *trace_file;
	char rw;
	uint32_t addr;

	if (argc != 2)
	{
	    printf("Error: Expected 1 command-line argument (trace file) but was provided %d.\n", (argc - 1));
	    exit(EXIT_FAILURE);
	}

	trace_file = argv[1];

	uint32_t fixed_blocksize = 32;

	int assoc_values[6] = {1, 2, 4, 8, 16, 32};
	int size_values[8] = {256, 512, 1024, 2048, 4096, 8192, 16384, 32768};
	int num_assoc = 6;
	int num_size = 8;
	const int num_configs = num_assoc * num_size;   // 48

	int max_assoc = 32;
	int max_size = 32768;
	int max_sets = max_size / (fixed_blocksize * 1);

	fp = fopen(trace_file, "r");
	if (fp == (FILE *)NULL) {
	    printf("Error: Unable to open file %s\n", trace_file);
	    exit(EXIT_FAILURE);
	}
	int total_no_of_ref = 0;
	{
	    char tmp_rw;
	    uint32_t tmp_addr;
	    while (fscanf(fp, "%c %x\n", &tmp_rw, &tmp_addr) == 2)
	    {
	        total_no_of_ref++;
	    }
	}
	rewind(fp);
	char* trace_rw = new char[total_no_of_ref];
	uint32_t* trace_addr = new uint32_t[total_no_of_ref];
	int idx = 0;
	while (fscanf(fp, "%c %x\n", &rw, &addr) == 2)
	{
	    trace_rw[idx] = rw;
	    trace_addr[idx] = addr;
	    idx++;
	}

	printf("===== Simulator configuration =====\n");
	printf("BLOCKSIZE:  %u\n", fixed_blocksize);
	printf("trace_file: %s\n", trace_file);
	printf("\n");

	int *g_l1_arr_valid, *g_l1_tag_storage, *g_l1_arr_lru;
	char *g_l1_arr_dirty;
	uint32_t *g_l1_arr_dirty_addr;
	cache_params *g_counters;
	uint32_t *g_assoc_values;
	char *g_trace_rw;
	uint32_t *g_trace_addr;

	size_t per_thread_size = max_sets * max_assoc;
	uint32_t *g_size_values;
	cudaMallocManaged(&g_size_values, num_size * sizeof(uint32_t));
	for (int i = 0; i < num_size; ++i) g_size_values[i] = size_values[i];

	cudaMallocManaged(&g_l1_arr_valid,      num_configs * per_thread_size * sizeof(int));
	cudaMallocManaged(&g_l1_tag_storage,    num_configs * per_thread_size * sizeof(int));
	cudaMallocManaged(&g_l1_arr_dirty,      num_configs * per_thread_size * sizeof(char));
	cudaMallocManaged(&g_l1_arr_dirty_addr, num_configs * per_thread_size * sizeof(uint32_t));
	cudaMallocManaged(&g_l1_arr_lru,        num_configs * per_thread_size * sizeof(int));
	cudaMallocManaged(&g_counters,          num_configs * sizeof(cache_params));
	cudaMallocManaged(&g_assoc_values,      num_assoc * sizeof(uint32_t));
	cudaMallocManaged(&g_trace_rw,          total_no_of_ref * sizeof(char));
	cudaMallocManaged(&g_trace_addr,        total_no_of_ref * sizeof(uint32_t));

	for (int i = 0; i < num_assoc; ++i) g_assoc_values[i] = assoc_values[i];
	for (int i = 0; i < total_no_of_ref; ++i) { g_trace_rw[i] = trace_rw[i]; g_trace_addr[i] = trace_addr[i]; }

	memset(g_l1_arr_valid, 0, num_configs * per_thread_size * sizeof(int));
	memset(g_l1_arr_dirty, ' ', num_configs * per_thread_size * sizeof(char));
	memset(g_l1_arr_dirty_addr, 0, num_configs * per_thread_size * sizeof(uint32_t));

	cudaEvent_t start, stop;
	cudaEventCreate(&start);
	cudaEventCreate(&stop);

	cudaEventRecord(start);
	int threadsPerBlock = 32;
	int numBlocks = (num_configs + threadsPerBlock - 1) / threadsPerBlock;
	sweep_kernel<<<numBlocks, threadsPerBlock>>>(num_configs, num_size, max_sets, max_assoc,
                                  g_assoc_values, g_size_values,
                                  g_l1_arr_valid, g_l1_tag_storage, g_l1_arr_dirty,
                                  g_l1_arr_dirty_addr, g_l1_arr_lru,
                                  g_counters,
                                  fixed_blocksize,
                                  g_trace_rw, g_trace_addr, total_no_of_ref);
	cudaEventRecord(stop);

	cudaEventSynchronize(stop);
	float kernel_ms = 0;
	cudaEventElapsedTime(&kernel_ms, start, stop);
	printf("Kernel execution time: %.3f ms\n", kernel_ms);

	cudaEventDestroy(start);
	cudaEventDestroy(stop);
	for (int i = 0; i < num_configs; ++i) {
	    cache_params &c = g_counters[i];
	    if (c.read_hit + c.write_hit + c.read_miss + c.write_miss == 0) {
	        printf("L1_ASSOC = %2u  L1_SIZE = %6u  ->  skipped (invalid: fewer than 1 set)\n", g_assoc_values[i / num_size], g_size_values[i % num_size]);
	    } else {
	        double miss_rate = static_cast<double>(c.write_miss + c.read_miss)
	            / (c.read_hit + c.write_hit + c.read_miss + c.write_miss);
	        printf("L1_ASSOC = %2u  L1_SIZE = %6u  ->  miss rate = %.4f\n",
	               g_assoc_values[i / num_size], g_size_values[i % num_size], miss_rate);
	    }
	}

	delete[] trace_rw;
	delete[] trace_addr;
	cudaFree(g_l1_arr_valid);
	cudaFree(g_l1_tag_storage);
	cudaFree(g_l1_arr_dirty);
	cudaFree(g_l1_arr_dirty_addr);
	cudaFree(g_l1_arr_lru);
	cudaFree(g_counters);
	cudaFree(g_assoc_values);
	cudaFree(g_trace_rw);
	cudaFree(g_trace_addr);
	cudaFree(g_size_values);

	return 0;
}

Overwriting cache_sim_gpu.cu


In [37]:
!nvcc --expt-relaxed-constexpr cache_sim_gpu.cu -o cache_sim_gpu_cuda
!./cache_sim_gpu_cuda gcc_trace.txt

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
===== Simulator configuration =====
BLOCKSIZE:  32
trace_file: gcc_trace.txt

Kernel execution time: 1619.331 ms
L1_ASSOC =  1  L1_SIZE =    256  ->  miss rate = 0.3336
L1_ASSOC =  1  L1_SIZE =    512  ->  miss rate = 0.2653
L1_ASSOC =  1  L1_SIZE =   1024  ->  miss rate = 0.1935
L1_ASSOC =  1  L1_SIZE =   2048  ->  miss rate = 0.1477
L1_ASSOC =  1  L1_SIZE =   4096  ->  miss rate = 0.1002
L1_ASSOC =  1  L1_SIZE =   8192  ->  miss rate = 0.0670
L1_ASSOC =  1  L1_SIZE =  16384  ->  miss rate = 0.0461
L1_ASSOC =  1  L1_SIZE =  32768  ->  miss rate = 0.0377
L1_ASSOC =  2  L1_SIZE =    256  ->  miss rate = 0.3053
L1_ASSOC =  2  L1_SIZE =    512  ->  miss rate = 0.2204
L1_ASSOC =  2  L1_SIZE =   1024  ->  miss rate = 0.1560
L1_ASSOC =  2  L1_SIZE =   2048  ->  miss rate = 0.1071
L1_ASSOC =  2  L1_SIZE =   4

CPU 192 Configs (associativity, block size and cache size changes)

In [38]:
%%writefile cache_sim_flat.cc
#include <stdio.h>
#include <stdlib.h>
#include <inttypes.h>
#include <cmath>
#include <tuple>
#include <cstdint>
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <chrono>
#include "cache_sim_flat.h"
using namespace std;
#include <cstring>

int main(int argc, char *argv[])
{
	FILE *fp;
	char *trace_file;
	char rw;
	uint32_t addr;

	if (argc != 2)
	{
	    printf("Error: Expected 1 command-line argument (trace file) but was provided %d.\n", (argc - 1));
	    exit(EXIT_FAILURE);
	}

	trace_file = argv[1];

	fp = fopen(trace_file, "r");
	if (fp == (FILE *)NULL) {
	    printf("Error: Unable to open file %s\n", trace_file);
	    exit(EXIT_FAILURE);
	}

	int total_no_of_ref = 0;
	{
	    char tmp_rw;
	    uint32_t tmp_addr;
	    while (fscanf(fp, "%c %x\n", &tmp_rw, &tmp_addr) == 2)
	    {
	        total_no_of_ref++;
	    }
	}
	rewind(fp);
	char* trace_rw = new char[total_no_of_ref];
	uint32_t* trace_addr = new uint32_t[total_no_of_ref];
	int idx = 0;
	while (fscanf(fp, "%c %x\n", &rw, &addr) == 2)
	{
	    trace_rw[idx] = rw;
	    trace_addr[idx] = addr;
	    idx++;
	}

	printf("===== Simulator configuration =====\n");
	printf("trace_file: %s\n", trace_file);
	printf("\n");
	int assoc_values[6] = {1, 2, 4, 8, 16, 32};
	int size_values[8] = {256, 512, 1024, 2048, 4096, 8192, 16384, 32768};
	int blocksize_values[4] = {8, 16, 32, 64};
	int num_assoc = 6;
	int num_size = 8;
	int num_blocksize = 4;
	int num_configs = num_assoc * num_size * num_blocksize;   // 192

	auto t0 = std::chrono::high_resolution_clock::now();

	for (int cfg = 0; cfg < num_configs; ++cfg)
	{
	    int assoc_idx    = cfg / (num_size * num_blocksize);
	    int size_idx     = (cfg / num_blocksize) % num_size;
	    int blocksize_idx= cfg % num_blocksize;

	    uint32_t this_assoc     = assoc_values[assoc_idx];
	    uint32_t this_size      = size_values[size_idx];
	    uint32_t this_blocksize = blocksize_values[blocksize_idx];

	    if (this_size < this_blocksize * this_assoc) {
	        printf("L1_ASSOC = %2u  L1_SIZE = %6u  BLOCKSIZE = %2u  ->  skipped (invalid: fewer than 1 set)\n",
	               this_assoc, this_size, this_blocksize);
	        continue;
	    }

	    int l1_no_of_sets = this_size / (this_blocksize * this_assoc);

	    int* l1_arr_valid = new int[l1_no_of_sets * this_assoc];
	    memset(l1_arr_valid, 0, l1_no_of_sets * this_assoc * sizeof(int));
	    char* l1_arr_dirty = new char[l1_no_of_sets * this_assoc];
	    memset(l1_arr_dirty, ' ', l1_no_of_sets * this_assoc * sizeof(char));
	    int* l1_tag_storage = new int[l1_no_of_sets * this_assoc];
	    memset(l1_tag_storage, 0, l1_no_of_sets * this_assoc * sizeof(int));
	    uint32_t* l1_arr_dirty_addr = new uint32_t[l1_no_of_sets * this_assoc];
	    memset(l1_arr_dirty_addr, 0, l1_no_of_sets * this_assoc * sizeof(uint32_t));
	    int* l1_arr_lru = new int[l1_no_of_sets * this_assoc];
	    for (int i = 0; i < l1_no_of_sets; ++i)
	        for (uint32_t j = 0; j < this_assoc; ++j)
	            l1_arr_lru[i * this_assoc + j] = j;

	    cache_params l1_counters;
	    cache_params l2_counters;

	    for (int i = 0; i < total_no_of_ref; ++i)
	    {
	        l1_cache(l1_tag_storage, l1_arr_dirty, l1_arr_valid, l1_arr_dirty_addr, l1_arr_lru,
	                 nullptr, nullptr, nullptr, nullptr, nullptr,
	                 l1_counters, l2_counters, this_blocksize, this_size, this_assoc,
	                 trace_addr[i], trace_rw[i], 0, 0,
	                 nullptr, nullptr, nullptr, nullptr, 0, 0, 0, 0);
	    }

	    double miss_rate = static_cast<double>(l1_counters.write_miss + l1_counters.read_miss)
	        / (l1_counters.read_hit + l1_counters.write_hit + l1_counters.read_miss + l1_counters.write_miss);

	    printf("L1_ASSOC = %2u  L1_SIZE = %6u  BLOCKSIZE = %2u  ->  L1 miss rate = %.4f\n",
	           this_assoc, this_size, this_blocksize, miss_rate);

	    delete[] l1_arr_valid;
	    delete[] l1_arr_dirty;
	    delete[] l1_tag_storage;
	    delete[] l1_arr_dirty_addr;
	    delete[] l1_arr_lru;
	}

	auto t1 = std::chrono::high_resolution_clock::now();
	double cpu_ms = std::chrono::duration<double, std::milli>(t1 - t0).count();
	printf("\nCPU sweep time: %.3f ms\n", cpu_ms);
	delete[] trace_rw;
	delete[] trace_addr;

	return 0;
}

Overwriting cache_sim_flat.cc


In [39]:
!g++ -std=c++17 -O2 -o cache_sim_flat cache_sim_flat.cc -I. && echo "Compiled successfully"
!time ./cache_sim_flat gcc_trace.txt

Compiled successfully
===== Simulator configuration =====
trace_file: gcc_trace.txt

L1_ASSOC =  1  L1_SIZE =    256  BLOCKSIZE =  8  ->  L1 miss rate = 0.3376
L1_ASSOC =  1  L1_SIZE =    256  BLOCKSIZE = 16  ->  L1 miss rate = 0.3026
L1_ASSOC =  1  L1_SIZE =    256  BLOCKSIZE = 32  ->  L1 miss rate = 0.3336
L1_ASSOC =  1  L1_SIZE =    256  BLOCKSIZE = 64  ->  L1 miss rate = 0.3769
L1_ASSOC =  1  L1_SIZE =    512  BLOCKSIZE =  8  ->  L1 miss rate = 0.2781
L1_ASSOC =  1  L1_SIZE =    512  BLOCKSIZE = 16  ->  L1 miss rate = 0.2454
L1_ASSOC =  1  L1_SIZE =    512  BLOCKSIZE = 32  ->  L1 miss rate = 0.2653
L1_ASSOC =  1  L1_SIZE =    512  BLOCKSIZE = 64  ->  L1 miss rate = 0.2872
L1_ASSOC =  1  L1_SIZE =   1024  BLOCKSIZE =  8  ->  L1 miss rate = 0.2280
L1_ASSOC =  1  L1_SIZE =   1024  BLOCKSIZE = 16  ->  L1 miss rate = 0.1922
L1_ASSOC =  1  L1_SIZE =   1024  BLOCKSIZE = 32  ->  L1 miss rate = 0.1935
L1_ASSOC =  1  L1_SIZE =   1024  BLOCKSIZE = 64  ->  L1 miss rate = 0.2149
L1_ASSOC =  1  

GPU 192 Threads (associativity, block size and cache size changes)

In [40]:
%%writefile cache_sim_gpu.cu
#include <stdio.h>
#include <stdlib.h>
#include <inttypes.h>
#include <cmath>
#include <tuple>
#include <cstdint>
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include "cache_sim_gpu.h"
using namespace std;
#include <cstring>

__global__ void sweep_kernel(int num_configs, int num_size, int num_blocksize, int max_sets, int max_assoc,
                              uint32_t* assoc_values_arr, uint32_t* size_values_arr, uint32_t* blocksize_values_arr,
                              int* l1_arr_valid, int* l1_tag_storage, char* l1_arr_dirty,
                              uint32_t* l1_arr_dirty_addr, int* l1_arr_lru,
                              cache_params* counters,
                              char* trace_rw, uint32_t* trace_addr, int total_no_of_ref)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid >= num_configs) return;

    int assoc_idx     = tid / (num_size * num_blocksize);
    int size_idx      = (tid / num_blocksize) % num_size;
    int blocksize_idx = tid % num_blocksize;

    uint32_t this_assoc     = assoc_values_arr[assoc_idx];
    uint32_t this_size      = size_values_arr[size_idx];
    uint32_t this_blocksize = blocksize_values_arr[blocksize_idx];

    int this_no_of_sets = this_size / (this_blocksize * this_assoc);
    size_t base = (size_t)tid * max_sets * max_assoc;

    int* my_valid       = l1_arr_valid + base;
    int* my_tags        = l1_tag_storage + base;
    char* my_dirty       = l1_arr_dirty + base;
    uint32_t* my_dirty_addr = l1_arr_dirty_addr + base;
    int* my_lru         = l1_arr_lru + base;

    cache_params my_counters;
    cache_params dummy_l2_counters;

    if (this_no_of_sets == 0) {
        counters[tid] = my_counters;
        return;
    }

    for (int s = 0; s < this_no_of_sets; ++s)
        for (uint32_t w = 0; w < this_assoc; ++w)
            my_lru[s * this_assoc + w] = w;

    for (int i = 0; i < total_no_of_ref; ++i)
    {
        l1_cache(my_tags, my_dirty, my_valid, my_dirty_addr, my_lru,
                 nullptr, nullptr, nullptr, nullptr, nullptr,
                 my_counters, dummy_l2_counters,
                 this_blocksize, this_size, this_assoc,
                 trace_addr[i], trace_rw[i], 0, 0,
                 nullptr, nullptr, nullptr, nullptr, 0, 0, 0, 0);
    }

    counters[tid] = my_counters;
}
int main(int argc, char *argv[])
{
	FILE *fp;
	char *trace_file;
	char rw;
	uint32_t addr;

	if (argc != 2)
	{
	    printf("Error: Expected 1 command-line argument (trace file) but was provided %d.\n", (argc - 1));
	    exit(EXIT_FAILURE);
	}

	trace_file = argv[1];

	int assoc_values[6] = {1, 2, 4, 8, 16, 32};
	int size_values[8] = {256, 512, 1024, 2048, 4096, 8192, 16384, 32768};
	int blocksize_values[4] = {8, 16, 32, 64};
	int num_assoc = 6;
	int num_size = 8;
	int num_blocksize = 4;
	const int num_configs = num_assoc * num_size * num_blocksize;   // 192

	int max_assoc = 32;
	int max_size = 32768;
	int min_blocksize = 8;
	int max_sets = max_size / (min_blocksize * 1);

	fp = fopen(trace_file, "r");
	if (fp == (FILE *)NULL) {
	    printf("Error: Unable to open file %s\n", trace_file);
	    exit(EXIT_FAILURE);
	}
	int total_no_of_ref = 0;
	{
	    char tmp_rw;
	    uint32_t tmp_addr;
	    while (fscanf(fp, "%c %x\n", &tmp_rw, &tmp_addr) == 2)
	    {
	        total_no_of_ref++;
	    }
	}
	rewind(fp);
	char* trace_rw = new char[total_no_of_ref];
	uint32_t* trace_addr = new uint32_t[total_no_of_ref];
	int idx = 0;
	while (fscanf(fp, "%c %x\n", &rw, &addr) == 2)
	{
	    trace_rw[idx] = rw;
	    trace_addr[idx] = addr;
	    idx++;
	}

	printf("===== Simulator configuration =====\n");
	printf("trace_file: %s\n", trace_file);
	printf("\n");

	int *g_l1_arr_valid, *g_l1_tag_storage, *g_l1_arr_lru;
	char *g_l1_arr_dirty;
	uint32_t *g_l1_arr_dirty_addr;
	cache_params *g_counters;
	uint32_t *g_assoc_values;
	char *g_trace_rw;
	uint32_t *g_trace_addr;

	size_t per_thread_size = max_sets * max_assoc;
	uint32_t *g_size_values;
	cudaMallocManaged(&g_size_values, num_size * sizeof(uint32_t));
	for (int i = 0; i < num_size; ++i) g_size_values[i] = size_values[i];
	uint32_t *g_blocksize_values;
	cudaMallocManaged(&g_blocksize_values, num_blocksize * sizeof(uint32_t));
	for (int i = 0; i < num_blocksize; ++i) g_blocksize_values[i] = blocksize_values[i];

	cudaMallocManaged(&g_l1_arr_valid,      num_configs * per_thread_size * sizeof(int));
	cudaMallocManaged(&g_l1_tag_storage,    num_configs * per_thread_size * sizeof(int));
	cudaMallocManaged(&g_l1_arr_dirty,      num_configs * per_thread_size * sizeof(char));
	cudaMallocManaged(&g_l1_arr_dirty_addr, num_configs * per_thread_size * sizeof(uint32_t));
	cudaMallocManaged(&g_l1_arr_lru,        num_configs * per_thread_size * sizeof(int));
	cudaMallocManaged(&g_counters,          num_configs * sizeof(cache_params));
	cudaMallocManaged(&g_assoc_values,      num_assoc * sizeof(uint32_t));
	cudaMallocManaged(&g_trace_rw,          total_no_of_ref * sizeof(char));
	cudaMallocManaged(&g_trace_addr,        total_no_of_ref * sizeof(uint32_t));

	for (int i = 0; i < num_assoc; ++i) g_assoc_values[i] = assoc_values[i];
	for (int i = 0; i < total_no_of_ref; ++i) { g_trace_rw[i] = trace_rw[i]; g_trace_addr[i] = trace_addr[i]; }

	memset(g_l1_arr_valid, 0, num_configs * per_thread_size * sizeof(int));
	memset(g_l1_arr_dirty, ' ', num_configs * per_thread_size * sizeof(char));
	memset(g_l1_arr_dirty_addr, 0, num_configs * per_thread_size * sizeof(uint32_t));

	cudaEvent_t start, stop;
	cudaEventCreate(&start);
	cudaEventCreate(&stop);

	cudaEventRecord(start);
	int threadsPerBlock = 32;
	int numBlocks = (num_configs + threadsPerBlock - 1) / threadsPerBlock;

	sweep_kernel<<<numBlocks, threadsPerBlock>>>(num_configs, num_size, num_blocksize, max_sets, max_assoc,
                                  g_assoc_values, g_size_values, g_blocksize_values,
                                  g_l1_arr_valid, g_l1_tag_storage, g_l1_arr_dirty,
                                  g_l1_arr_dirty_addr, g_l1_arr_lru,
                                  g_counters,
                                  g_trace_rw, g_trace_addr, total_no_of_ref);
	cudaEventRecord(stop);

	cudaEventSynchronize(stop);
	float kernel_ms = 0;
	cudaEventElapsedTime(&kernel_ms, start, stop);
	printf("Kernel execution time: %.3f ms\n", kernel_ms);

	cudaEventDestroy(start);
	cudaEventDestroy(stop);
	for (int i = 0; i < num_configs; ++i) {
	    cache_params &c = g_counters[i];
	    int assoc_idx     = i / (num_size * num_blocksize);
	    int size_idx      = (i / num_blocksize) % num_size;
	    int blocksize_idx = i % num_blocksize;
	    uint32_t a = g_assoc_values[assoc_idx];
	    uint32_t s = g_size_values[size_idx];
	    uint32_t b = g_blocksize_values[blocksize_idx];

	    int total = c.read_hit + c.write_hit + c.read_miss + c.write_miss;
	    if (total == 0) {
	        printf("L1_ASSOC = %2u  L1_SIZE = %6u  BLOCKSIZE = %2u  ->  skipped (invalid: fewer than 1 set)\n", a, s, b);
	    } else {
	        double miss_rate = static_cast<double>(c.write_miss + c.read_miss) / total;
	        printf("L1_ASSOC = %2u  L1_SIZE = %6u  BLOCKSIZE = %2u  ->  miss rate = %.4f\n", a, s, b, miss_rate);
	    }
	}

	delete[] trace_rw;
	delete[] trace_addr;
	cudaFree(g_l1_arr_valid);
	cudaFree(g_l1_tag_storage);
	cudaFree(g_l1_arr_dirty);
	cudaFree(g_l1_arr_dirty_addr);
	cudaFree(g_l1_arr_lru);
	cudaFree(g_counters);
	cudaFree(g_assoc_values);
	cudaFree(g_trace_rw);
	cudaFree(g_trace_addr);
	cudaFree(g_size_values);
	cudaFree(g_blocksize_values);

	return 0;
}

Overwriting cache_sim_gpu.cu


In [41]:
!nvcc --expt-relaxed-constexpr cache_sim_gpu.cu -o cache_sim_gpu_cuda
!./cache_sim_gpu_cuda gcc_trace.txt

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
===== Simulator configuration =====
trace_file: gcc_trace.txt

Kernel execution time: 3180.021 ms
L1_ASSOC =  1  L1_SIZE =    256  BLOCKSIZE =  8  ->  miss rate = 0.3376
L1_ASSOC =  1  L1_SIZE =    256  BLOCKSIZE = 16  ->  miss rate = 0.3026
L1_ASSOC =  1  L1_SIZE =    256  BLOCKSIZE = 32  ->  miss rate = 0.3336
L1_ASSOC =  1  L1_SIZE =    256  BLOCKSIZE = 64  ->  miss rate = 0.3769
L1_ASSOC =  1  L1_SIZE =    512  BLOCKSIZE =  8  ->  miss rate = 0.2781
L1_ASSOC =  1  L1_SIZE =    512  BLOCKSIZE = 16  ->  miss rate = 0.2454
L1_ASSOC =  1  L1_SIZE =    512  BLOCKSIZE = 32  ->  miss rate = 0.2653
L1_ASSOC =  1  L1_SIZE =    512  BLOCKSIZE = 64  ->  miss rate = 0.2872
L1_ASSOC =  1  L1_SIZE =   1024  BLOCKSIZE =  8  ->  miss rate = 0.2280
L1_ASSOC =  1  L1_SIZE =   1024  BLOCKSIZE = 16  ->  miss rate = 0.

CPU 448 threads (increased the number of block sizes and associativity values from before to increase the number of threads)

In [49]:
%%writefile cache_sim_flat.cc
#include <stdio.h>
#include <stdlib.h>
#include <inttypes.h>
#include <cmath>
#include <tuple>
#include <cstdint>
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <chrono>
#include "cache_sim_flat.h"
using namespace std;
#include <cstring>

int main(int argc, char *argv[])
{
	FILE *fp;
	char *trace_file;
	char rw;
	uint32_t addr;

	if (argc != 2)
	{
	    printf("Error: Expected 1 command-line argument (trace file) but was provided %d.\n", (argc - 1));
	    exit(EXIT_FAILURE);
	}

	trace_file = argv[1];

	fp = fopen(trace_file, "r");
	if (fp == (FILE *)NULL) {
	    printf("Error: Unable to open file %s\n", trace_file);
	    exit(EXIT_FAILURE);
	}

	int total_no_of_ref = 0;
	{
	    char tmp_rw;
	    uint32_t tmp_addr;
	    while (fscanf(fp, "%c %x\n", &tmp_rw, &tmp_addr) == 2)
	    {
	        total_no_of_ref++;
	    }
	}
	rewind(fp);
	char* trace_rw = new char[total_no_of_ref];
	uint32_t* trace_addr = new uint32_t[total_no_of_ref];
	int idx = 0;
	while (fscanf(fp, "%c %x\n", &rw, &addr) == 2)
	{
	    trace_rw[idx] = rw;
	    trace_addr[idx] = addr;
	    idx++;
	}

	printf("===== Simulator configuration =====\n");
	printf("trace_file: %s\n", trace_file);
	printf("\n");
	int assoc_values[8] = {1, 2, 4, 8, 16, 32, 64, 128};
	int size_values[8] = {256, 512, 1024, 2048, 4096, 8192, 16384, 32768};
	int blocksize_values[7] = {4, 8, 16, 32, 64, 128, 256};
	int num_assoc = 8;
	int num_size = 8;
	int num_blocksize = 7;
	int num_configs = num_assoc * num_size * num_blocksize;   // 448

	auto t0 = std::chrono::high_resolution_clock::now();

	for (int cfg = 0; cfg < num_configs; ++cfg)
	{
	    int assoc_idx    = cfg / (num_size * num_blocksize);
	    int size_idx     = (cfg / num_blocksize) % num_size;
	    int blocksize_idx= cfg % num_blocksize;

	    uint32_t this_assoc     = assoc_values[assoc_idx];
	    uint32_t this_size      = size_values[size_idx];
	    uint32_t this_blocksize = blocksize_values[blocksize_idx];

	    if (this_size < this_blocksize * this_assoc) {
	        printf("L1_ASSOC = %2u  L1_SIZE = %6u  BLOCKSIZE = %2u  ->  skipped (invalid: fewer than 1 set)\n",
	               this_assoc, this_size, this_blocksize);
	        continue;
	    }

	    int l1_no_of_sets = this_size / (this_blocksize * this_assoc);

	    int* l1_arr_valid = new int[l1_no_of_sets * this_assoc];
	    memset(l1_arr_valid, 0, l1_no_of_sets * this_assoc * sizeof(int));
	    char* l1_arr_dirty = new char[l1_no_of_sets * this_assoc];
	    memset(l1_arr_dirty, ' ', l1_no_of_sets * this_assoc * sizeof(char));
	    int* l1_tag_storage = new int[l1_no_of_sets * this_assoc];
	    memset(l1_tag_storage, 0, l1_no_of_sets * this_assoc * sizeof(int));
	    uint32_t* l1_arr_dirty_addr = new uint32_t[l1_no_of_sets * this_assoc];
	    memset(l1_arr_dirty_addr, 0, l1_no_of_sets * this_assoc * sizeof(uint32_t));
	    int* l1_arr_lru = new int[l1_no_of_sets * this_assoc];
	    for (int i = 0; i < l1_no_of_sets; ++i)
	        for (uint32_t j = 0; j < this_assoc; ++j)
	            l1_arr_lru[i * this_assoc + j] = j;

	    cache_params l1_counters;
	    cache_params l2_counters;

	    for (int i = 0; i < total_no_of_ref; ++i)
	    {
	        l1_cache(l1_tag_storage, l1_arr_dirty, l1_arr_valid, l1_arr_dirty_addr, l1_arr_lru,
	                 nullptr, nullptr, nullptr, nullptr, nullptr,
	                 l1_counters, l2_counters, this_blocksize, this_size, this_assoc,
	                 trace_addr[i], trace_rw[i], 0, 0,
	                 nullptr, nullptr, nullptr, nullptr, 0, 0, 0, 0);
	    }

	    double miss_rate = static_cast<double>(l1_counters.write_miss + l1_counters.read_miss)
	        / (l1_counters.read_hit + l1_counters.write_hit + l1_counters.read_miss + l1_counters.write_miss);

	    printf("L1_ASSOC = %2u  L1_SIZE = %6u  BLOCKSIZE = %2u  ->  L1 miss rate = %.4f\n",
	           this_assoc, this_size, this_blocksize, miss_rate);

	    delete[] l1_arr_valid;
	    delete[] l1_arr_dirty;
	    delete[] l1_tag_storage;
	    delete[] l1_arr_dirty_addr;
	    delete[] l1_arr_lru;
	}

	auto t1 = std::chrono::high_resolution_clock::now();
	double cpu_ms = std::chrono::duration<double, std::milli>(t1 - t0).count();
	printf("\nCPU sweep time: %.3f ms\n", cpu_ms);
	delete[] trace_rw;
	delete[] trace_addr;

	return 0;
}

Overwriting cache_sim_flat.cc


In [50]:
!g++ -std=c++17 -O2 -o cache_sim_flat cache_sim_flat.cc -I. && echo "Compiled successfully"
!time ./cache_sim_flat gcc_trace.txt

Compiled successfully
===== Simulator configuration =====
trace_file: gcc_trace.txt

L1_ASSOC =  1  L1_SIZE =    256  BLOCKSIZE =  4  ->  L1 miss rate = 0.4300
L1_ASSOC =  1  L1_SIZE =    256  BLOCKSIZE =  8  ->  L1 miss rate = 0.3376
L1_ASSOC =  1  L1_SIZE =    256  BLOCKSIZE = 16  ->  L1 miss rate = 0.3026
L1_ASSOC =  1  L1_SIZE =    256  BLOCKSIZE = 32  ->  L1 miss rate = 0.3336
L1_ASSOC =  1  L1_SIZE =    256  BLOCKSIZE = 64  ->  L1 miss rate = 0.3769
L1_ASSOC =  1  L1_SIZE =    256  BLOCKSIZE = 128  ->  L1 miss rate = 0.4295
L1_ASSOC =  1  L1_SIZE =    256  BLOCKSIZE = 256  ->  L1 miss rate = 0.5096
L1_ASSOC =  1  L1_SIZE =    512  BLOCKSIZE =  4  ->  L1 miss rate = 0.3606
L1_ASSOC =  1  L1_SIZE =    512  BLOCKSIZE =  8  ->  L1 miss rate = 0.2781
L1_ASSOC =  1  L1_SIZE =    512  BLOCKSIZE = 16  ->  L1 miss rate = 0.2454
L1_ASSOC =  1  L1_SIZE =    512  BLOCKSIZE = 32  ->  L1 miss rate = 0.2653
L1_ASSOC =  1  L1_SIZE =    512  BLOCKSIZE = 64  ->  L1 miss rate = 0.2872
L1_ASSOC =  1

GPU 448 Threads (increased the number of block sizes and associativity values from before to increase the number of threads)

In [51]:
%%writefile cache_sim_gpu.cu
#include <stdio.h>
#include <stdlib.h>
#include <inttypes.h>
#include <cmath>
#include <tuple>
#include <cstdint>
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include "cache_sim_gpu.h"
using namespace std;
#include <cstring>

__global__ void sweep_kernel(int num_configs, int num_size, int num_blocksize, int max_sets, int max_assoc,
                              uint32_t* assoc_values_arr, uint32_t* size_values_arr, uint32_t* blocksize_values_arr,
                              int* l1_arr_valid, int* l1_tag_storage, char* l1_arr_dirty,
                              uint32_t* l1_arr_dirty_addr, int* l1_arr_lru,
                              cache_params* counters,
                              char* trace_rw, uint32_t* trace_addr, int total_no_of_ref)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid >= num_configs) return;

    int assoc_idx     = tid / (num_size * num_blocksize);
    int size_idx      = (tid / num_blocksize) % num_size;
    int blocksize_idx = tid % num_blocksize;

    uint32_t this_assoc     = assoc_values_arr[assoc_idx];
    uint32_t this_size      = size_values_arr[size_idx];
    uint32_t this_blocksize = blocksize_values_arr[blocksize_idx];

    int this_no_of_sets = this_size / (this_blocksize * this_assoc);
    size_t base = (size_t)tid * max_sets * max_assoc;

    int* my_valid       = l1_arr_valid + base;
    int* my_tags        = l1_tag_storage + base;
    char* my_dirty       = l1_arr_dirty + base;
    uint32_t* my_dirty_addr = l1_arr_dirty_addr + base;
    int* my_lru         = l1_arr_lru + base;

    cache_params my_counters;
    cache_params dummy_l2_counters;

    if (this_no_of_sets == 0) {
        counters[tid] = my_counters;
        return;
    }

    for (int s = 0; s < this_no_of_sets; ++s)
        for (uint32_t w = 0; w < this_assoc; ++w)
            my_lru[s * this_assoc + w] = w;

    for (int i = 0; i < total_no_of_ref; ++i)
    {
        l1_cache(my_tags, my_dirty, my_valid, my_dirty_addr, my_lru,
                 nullptr, nullptr, nullptr, nullptr, nullptr,
                 my_counters, dummy_l2_counters,
                 this_blocksize, this_size, this_assoc,
                 trace_addr[i], trace_rw[i], 0, 0,
                 nullptr, nullptr, nullptr, nullptr, 0, 0, 0, 0);
    }

    counters[tid] = my_counters;
}
int main(int argc, char *argv[])
{
	FILE *fp;
	char *trace_file;
	char rw;
	uint32_t addr;

	if (argc != 2)
	{
	    printf("Error: Expected 1 command-line argument (trace file) but was provided %d.\n", (argc - 1));
	    exit(EXIT_FAILURE);
	}

	trace_file = argv[1];

	int assoc_values[8] = {1, 2, 4, 8, 16, 32, 64, 128};
	int size_values[8] = {256, 512, 1024, 2048, 4096, 8192, 16384, 32768};
	int blocksize_values[7] = {4, 8, 16, 32, 64, 128, 256};
	int num_assoc = 8;
	int num_size = 8;
	int num_blocksize = 7;
	const int num_configs = num_assoc * num_size * num_blocksize;   // 448

	int max_assoc = 128;
	int max_size = 32768;
	int min_blocksize = 4;
	int max_sets = max_size / (min_blocksize * 1);

	fp = fopen(trace_file, "r");
	if (fp == (FILE *)NULL) {
	    printf("Error: Unable to open file %s\n", trace_file);
	    exit(EXIT_FAILURE);
	}
	int total_no_of_ref = 0;
	{
	    char tmp_rw;
	    uint32_t tmp_addr;
	    while (fscanf(fp, "%c %x\n", &tmp_rw, &tmp_addr) == 2)
	    {
	        total_no_of_ref++;
	    }
	}
	rewind(fp);
	char* trace_rw = new char[total_no_of_ref];
	uint32_t* trace_addr = new uint32_t[total_no_of_ref];
	int idx = 0;
	while (fscanf(fp, "%c %x\n", &rw, &addr) == 2)
	{
	    trace_rw[idx] = rw;
	    trace_addr[idx] = addr;
	    idx++;
	}

	printf("===== Simulator configuration =====\n");
	printf("trace_file: %s\n", trace_file);
	printf("\n");

	int *g_l1_arr_valid, *g_l1_tag_storage, *g_l1_arr_lru;
	char *g_l1_arr_dirty;
	uint32_t *g_l1_arr_dirty_addr;
	cache_params *g_counters;
	uint32_t *g_assoc_values;
	char *g_trace_rw;
	uint32_t *g_trace_addr;

	size_t per_thread_size = max_sets * max_assoc;
	uint32_t *g_size_values;
	cudaMallocManaged(&g_size_values, num_size * sizeof(uint32_t));
	for (int i = 0; i < num_size; ++i) g_size_values[i] = size_values[i];
	uint32_t *g_blocksize_values;
	cudaMallocManaged(&g_blocksize_values, num_blocksize * sizeof(uint32_t));
	for (int i = 0; i < num_blocksize; ++i) g_blocksize_values[i] = blocksize_values[i];

	cudaMallocManaged(&g_l1_arr_valid,      num_configs * per_thread_size * sizeof(int));
	cudaMallocManaged(&g_l1_tag_storage,    num_configs * per_thread_size * sizeof(int));
	cudaMallocManaged(&g_l1_arr_dirty,      num_configs * per_thread_size * sizeof(char));
	cudaMallocManaged(&g_l1_arr_dirty_addr, num_configs * per_thread_size * sizeof(uint32_t));
	cudaMallocManaged(&g_l1_arr_lru,        num_configs * per_thread_size * sizeof(int));
	cudaMallocManaged(&g_counters,          num_configs * sizeof(cache_params));
	cudaMallocManaged(&g_assoc_values,      num_assoc * sizeof(uint32_t));
	cudaMallocManaged(&g_trace_rw,          total_no_of_ref * sizeof(char));
	cudaMallocManaged(&g_trace_addr,        total_no_of_ref * sizeof(uint32_t));

	for (int i = 0; i < num_assoc; ++i) g_assoc_values[i] = assoc_values[i];
	for (int i = 0; i < total_no_of_ref; ++i) { g_trace_rw[i] = trace_rw[i]; g_trace_addr[i] = trace_addr[i]; }

	memset(g_l1_arr_valid, 0, num_configs * per_thread_size * sizeof(int));
	memset(g_l1_arr_dirty, ' ', num_configs * per_thread_size * sizeof(char));
	memset(g_l1_arr_dirty_addr, 0, num_configs * per_thread_size * sizeof(uint32_t));

	cudaEvent_t start, stop;
	cudaEventCreate(&start);
	cudaEventCreate(&stop);

	cudaEventRecord(start);
	int threadsPerBlock = 32;
	int numBlocks = (num_configs + threadsPerBlock - 1) / threadsPerBlock;

	sweep_kernel<<<numBlocks, threadsPerBlock>>>(num_configs, num_size, num_blocksize, max_sets, max_assoc,
                                  g_assoc_values, g_size_values, g_blocksize_values,
                                  g_l1_arr_valid, g_l1_tag_storage, g_l1_arr_dirty,
                                  g_l1_arr_dirty_addr, g_l1_arr_lru,
                                  g_counters,
                                  g_trace_rw, g_trace_addr, total_no_of_ref);
	cudaEventRecord(stop);

	cudaEventSynchronize(stop);
	float kernel_ms = 0;
	cudaEventElapsedTime(&kernel_ms, start, stop);
	printf("Kernel execution time: %.3f ms\n", kernel_ms);

	cudaEventDestroy(start);
	cudaEventDestroy(stop);
	for (int i = 0; i < num_configs; ++i) {
	    cache_params &c = g_counters[i];
	    int assoc_idx     = i / (num_size * num_blocksize);
	    int size_idx      = (i / num_blocksize) % num_size;
	    int blocksize_idx = i % num_blocksize;
	    uint32_t a = g_assoc_values[assoc_idx];
	    uint32_t s = g_size_values[size_idx];
	    uint32_t b = g_blocksize_values[blocksize_idx];

	    int total = c.read_hit + c.write_hit + c.read_miss + c.write_miss;
	    if (total == 0) {
	        printf("L1_ASSOC = %2u  L1_SIZE = %6u  BLOCKSIZE = %2u  ->  skipped (invalid: fewer than 1 set)\n", a, s, b);
	    } else {
	        double miss_rate = static_cast<double>(c.write_miss + c.read_miss) / total;
	        printf("L1_ASSOC = %2u  L1_SIZE = %6u  BLOCKSIZE = %2u  ->  L1 miss rate = %.4f\n", a, s, b, miss_rate);
	    }
	}

	delete[] trace_rw;
	delete[] trace_addr;
	cudaFree(g_l1_arr_valid);
	cudaFree(g_l1_tag_storage);
	cudaFree(g_l1_arr_dirty);
	cudaFree(g_l1_arr_dirty_addr);
	cudaFree(g_l1_arr_lru);
	cudaFree(g_counters);
	cudaFree(g_assoc_values);
	cudaFree(g_trace_rw);
	cudaFree(g_trace_addr);
	cudaFree(g_size_values);
	cudaFree(g_blocksize_values);

	return 0;
}

Overwriting cache_sim_gpu.cu


In [52]:
!nvcc --expt-relaxed-constexpr cache_sim_gpu.cu -o cache_sim_gpu_cuda
!./cache_sim_gpu_cuda gcc_trace.txt

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
===== Simulator configuration =====
trace_file: gcc_trace.txt

Kernel execution time: 12801.156 ms
L1_ASSOC =  1  L1_SIZE =    256  BLOCKSIZE =  4  ->  L1 miss rate = 0.4300
L1_ASSOC =  1  L1_SIZE =    256  BLOCKSIZE =  8  ->  L1 miss rate = 0.3376
L1_ASSOC =  1  L1_SIZE =    256  BLOCKSIZE = 16  ->  L1 miss rate = 0.3026
L1_ASSOC =  1  L1_SIZE =    256  BLOCKSIZE = 32  ->  L1 miss rate = 0.3336
L1_ASSOC =  1  L1_SIZE =    256  BLOCKSIZE = 64  ->  L1 miss rate = 0.3769
L1_ASSOC =  1  L1_SIZE =    256  BLOCKSIZE = 128  ->  L1 miss rate = 0.4295
L1_ASSOC =  1  L1_SIZE =    256  BLOCKSIZE = 256  ->  L1 miss rate = 0.5096
L1_ASSOC =  1  L1_SIZE =    512  BLOCKSIZE =  4  ->  L1 miss rate = 0.3606
L1_ASSOC =  1  L1_SIZE =    512  BLOCKSIZE =  8  ->  L1 miss rate = 0.2781
L1_ASSOC =  1  L1_SIZE =    512  BLOC

It was observed that the time was getting saturated with CPU time being approximately 4 times faster than GPU, hence I generalized pref_M and pref_N values to increase the thread count to 648

CPU 648 Threads

In [8]:
%%writefile cache_sim_flat.cc
#include <stdio.h>
#include <stdlib.h>
#include <inttypes.h>
#include <cmath>
#include <tuple>
#include <cstdint>
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <chrono>
#include "cache_sim_flat.h"
using namespace std;
#include <cstring>

int main(int argc, char *argv[])
{
	FILE *fp;
	char *trace_file;
	char rw;
	uint32_t addr;

	if (argc != 2)
	{
	    printf("Error: Expected 1 command-line argument (trace file) but was provided %d.\n", (argc - 1));
	    exit(EXIT_FAILURE);
	}

	trace_file = argv[1];
	fp = fopen(trace_file, "r");
	if (fp == (FILE *)NULL) {
	    printf("Error: Unable to open file %s\n", trace_file);
	    exit(EXIT_FAILURE);
	}

	int total_no_of_ref = 0;
	{
	    char tmp_rw;
	    uint32_t tmp_addr;
	    while (fscanf(fp, "%c %x\n", &tmp_rw, &tmp_addr) == 2)
	    {
	        total_no_of_ref++;
	    }
	}
	rewind(fp);
	char* trace_rw = new char[total_no_of_ref];
	uint32_t* trace_addr = new uint32_t[total_no_of_ref];
	int idx = 0;
	while (fscanf(fp, "%c %x\n", &rw, &addr) == 2)
	{
	    trace_rw[idx] = rw;
	    trace_addr[idx] = addr;
	    idx++;
	}

	printf("===== Simulator configuration =====\n");
	printf("trace_file: %s\n\n", trace_file);

	int assoc_values[6] = {1, 2, 4, 8, 16, 32};
	int size_values[6] = {1024, 2048, 4096, 8192, 16384, 32768};
	int blocksize_values[3] = {16, 32, 64};
	int pref_n_values[3] = {0, 4, 8};
	int pref_m_values[2] = {2, 4};
	int num_assoc = 6;
	int num_size = 6;
	int num_blocksize = 3;
	int num_pref_n = 3;
	int num_pref_m = 2;
	int num_configs = num_assoc * num_size * num_blocksize * num_pref_n * num_pref_m;  // 648

	auto t0 = std::chrono::high_resolution_clock::now();

	for (int cfg = 0; cfg < num_configs; ++cfg)
	{
	    int pref_m_idx    = cfg % num_pref_m;
	    int pref_n_idx    = (cfg / num_pref_m) % num_pref_n;
	    int blocksize_idx = (cfg / (num_pref_m * num_pref_n)) % num_blocksize;
	    int size_idx      = (cfg / (num_pref_m * num_pref_n * num_blocksize)) % num_size;
	    int assoc_idx     = cfg / (num_pref_m * num_pref_n * num_blocksize * num_size);

	    uint32_t this_assoc     = assoc_values[assoc_idx];
	    uint32_t this_size      = size_values[size_idx];
	    uint32_t this_blocksize = blocksize_values[blocksize_idx];
	    uint32_t this_pref_n    = pref_n_values[pref_n_idx];
	    uint32_t this_pref_m    = pref_m_values[pref_m_idx];

	    if (this_size < this_blocksize * this_assoc) {
	        printf("ASSOC=%2u SIZE=%6u BLK=%2u PREF_N=%u PREF_M=%u -> skipped (invalid: fewer than 1 set)\n",
	               this_assoc, this_size, this_blocksize, this_pref_n, this_pref_m);
	        continue;
	    }

	    int l1_no_of_sets = this_size / (this_blocksize * this_assoc);

	    int* l1_arr_valid = new int[l1_no_of_sets * this_assoc];
	    memset(l1_arr_valid, 0, l1_no_of_sets * this_assoc * sizeof(int));
	    char* l1_arr_dirty = new char[l1_no_of_sets * this_assoc];
	    memset(l1_arr_dirty, ' ', l1_no_of_sets * this_assoc * sizeof(char));
	    int* l1_tag_storage = new int[l1_no_of_sets * this_assoc];
	    memset(l1_tag_storage, 0, l1_no_of_sets * this_assoc * sizeof(int));
	    uint32_t* l1_arr_dirty_addr = new uint32_t[l1_no_of_sets * this_assoc];
	    memset(l1_arr_dirty_addr, 0, l1_no_of_sets * this_assoc * sizeof(uint32_t));
	    int* l1_arr_lru = new int[l1_no_of_sets * this_assoc];
	    for (int i = 0; i < l1_no_of_sets; ++i)
	        for (uint32_t j = 0; j < this_assoc; ++j)
	            l1_arr_lru[i * this_assoc + j] = j;

	    // per-config stream buffer arrays (only meaningful if this_pref_n > 0)
	    bool* sb_valid = nullptr;
	    int* sb_top = nullptr;
	    int* sb_rec = nullptr;
	    uint32_t* sb_blocks = nullptr;
	    if (this_pref_n > 0) {
	        sb_valid = new bool[this_pref_n];
	        memset(sb_valid, false, this_pref_n * sizeof(bool));
	        sb_top = new int[this_pref_n];
	        memset(sb_top, 0, this_pref_n * sizeof(int));
	        sb_rec = new int[this_pref_n];
	        for (uint32_t i = 0; i < this_pref_n; ++i) sb_rec[i] = this_pref_n - i;
	        sb_blocks = new uint32_t[this_pref_n * this_pref_m];
	        memset(sb_blocks, 0, this_pref_n * this_pref_m * sizeof(uint32_t));
	    }

	    cache_params l1_counters;
	    cache_params l2_counters;

	    for (int i = 0; i < total_no_of_ref; ++i)
	    {
	        l1_cache(l1_tag_storage, l1_arr_dirty, l1_arr_valid, l1_arr_dirty_addr, l1_arr_lru,
	                 nullptr, nullptr, nullptr, nullptr, nullptr,
	                 l1_counters, l2_counters, this_blocksize, this_size, this_assoc,
	                 trace_addr[i], trace_rw[i], 0, 0,
	                 sb_valid, sb_top, sb_rec, sb_blocks,
	                 this_pref_n, this_pref_m, 0, 0);
	    }

	    double miss_rate = static_cast<double>(l1_counters.write_miss + l1_counters.read_miss)
	        / (l1_counters.read_hit + l1_counters.write_hit + l1_counters.read_miss + l1_counters.write_miss
	           + l1_counters.sb_read_hits + l1_counters.sb_write_hits);

	    printf("ASSOC=%2u SIZE=%6u BLK=%2u PREF_N=%u PREF_M=%u -> L1 miss rate = %.4f\n",
	           this_assoc, this_size, this_blocksize, this_pref_n, this_pref_m, miss_rate);

	    delete[] l1_arr_valid;
	    delete[] l1_arr_dirty;
	    delete[] l1_tag_storage;
	    delete[] l1_arr_dirty_addr;
	    delete[] l1_arr_lru;
	    if (sb_valid) delete[] sb_valid;
	    if (sb_top) delete[] sb_top;
	    if (sb_rec) delete[] sb_rec;
	    if (sb_blocks) delete[] sb_blocks;
	}

	auto t1 = std::chrono::high_resolution_clock::now();
	double cpu_ms = std::chrono::duration<double, std::milli>(t1 - t0).count();
	printf("\nCPU sweep time: %.3f ms\n", cpu_ms);

	delete[] trace_rw;
	delete[] trace_addr;

	return 0;
}

Overwriting cache_sim_flat.cc


In [9]:
!g++ -std=c++17 -O2 -o cache_sim_flat cache_sim_flat.cc -I. && echo "Compiled successfully"
!time ./cache_sim_flat gcc_trace.txt

Compiled successfully
===== Simulator configuration =====
trace_file: gcc_trace.txt

ASSOC= 1 SIZE=  1024 BLK=16 PREF_N=0 PREF_M=2 -> L1 miss rate = 0.1922
ASSOC= 1 SIZE=  1024 BLK=16 PREF_N=0 PREF_M=4 -> L1 miss rate = 0.1922
ASSOC= 1 SIZE=  1024 BLK=16 PREF_N=4 PREF_M=2 -> L1 miss rate = 0.1436
ASSOC= 1 SIZE=  1024 BLK=16 PREF_N=4 PREF_M=4 -> L1 miss rate = 0.1429
ASSOC= 1 SIZE=  1024 BLK=16 PREF_N=8 PREF_M=2 -> L1 miss rate = 0.1369
ASSOC= 1 SIZE=  1024 BLK=16 PREF_N=8 PREF_M=4 -> L1 miss rate = 0.1349
ASSOC= 1 SIZE=  1024 BLK=32 PREF_N=0 PREF_M=2 -> L1 miss rate = 0.1935
ASSOC= 1 SIZE=  1024 BLK=32 PREF_N=0 PREF_M=4 -> L1 miss rate = 0.1935
ASSOC= 1 SIZE=  1024 BLK=32 PREF_N=4 PREF_M=2 -> L1 miss rate = 0.1639
ASSOC= 1 SIZE=  1024 BLK=32 PREF_N=4 PREF_M=4 -> L1 miss rate = 0.1620
ASSOC= 1 SIZE=  1024 BLK=32 PREF_N=8 PREF_M=2 -> L1 miss rate = 0.1579
ASSOC= 1 SIZE=  1024 BLK=32 PREF_N=8 PREF_M=4 -> L1 miss rate = 0.1547
ASSOC= 1 SIZE=  1024 BLK=64 PREF_N=0 PREF_M=2 -> L1 miss rate =

GPU 648 Threads

In [10]:
%%writefile cache_sim_gpu.cu
#include <stdio.h>
#include <stdlib.h>
#include <inttypes.h>
#include <cmath>
#include <tuple>
#include <cstdint>
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include "cache_sim_gpu.h"
using namespace std;
#include <cstring>

__global__ void sweep_kernel(int num_configs, int num_size, int num_blocksize, int num_pref_n, int num_pref_m,
                              int max_sets, int max_assoc, int max_pref_n, int max_pref_m,
                              uint32_t* assoc_values_arr, uint32_t* size_values_arr, uint32_t* blocksize_values_arr,
                              uint32_t* pref_n_values_arr, uint32_t* pref_m_values_arr,
                              int* l1_arr_valid, int* l1_tag_storage, char* l1_arr_dirty,
                              uint32_t* l1_arr_dirty_addr, int* l1_arr_lru,
                              bool* sb_valid_g, int* sb_top_g, int* sb_rec_g, uint32_t* sb_blocks_g,
                              cache_params* counters,
                              char* trace_rw, uint32_t* trace_addr, int total_no_of_ref)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid >= num_configs) return;

    int pref_m_idx    = tid % num_pref_m;
    int pref_n_idx    = (tid / num_pref_m) % num_pref_n;
    int blocksize_idx = (tid / (num_pref_m * num_pref_n)) % num_blocksize;
    int size_idx      = (tid / (num_pref_m * num_pref_n * num_blocksize)) % num_size;
    int assoc_idx     = tid / (num_pref_m * num_pref_n * num_blocksize * num_size);

    uint32_t this_assoc     = assoc_values_arr[assoc_idx];
    uint32_t this_size      = size_values_arr[size_idx];
    uint32_t this_blocksize = blocksize_values_arr[blocksize_idx];
    uint32_t this_pref_n    = pref_n_values_arr[pref_n_idx];
    uint32_t this_pref_m    = pref_m_values_arr[pref_m_idx];

    int this_no_of_sets = this_size / (this_blocksize * this_assoc);
    size_t base           = (size_t)tid * max_sets * max_assoc;
    size_t sb_base         = (size_t)tid * max_pref_n;
    size_t sb_blocks_base  = (size_t)tid * max_pref_n * max_pref_m;

    int* my_valid       = l1_arr_valid + base;
    int* my_tags        = l1_tag_storage + base;
    char* my_dirty       = l1_arr_dirty + base;
    uint32_t* my_dirty_addr = l1_arr_dirty_addr + base;
    int* my_lru         = l1_arr_lru + base;

    bool* my_sb_valid      = sb_valid_g + sb_base;
    int* my_sb_top         = sb_top_g + sb_base;
    int* my_sb_rec         = sb_rec_g + sb_base;
    uint32_t* my_sb_blocks = sb_blocks_g + sb_blocks_base;

    cache_params my_counters;
    cache_params dummy_l2_counters;

    if (this_no_of_sets == 0) {
        counters[tid] = my_counters;
        return;
    }

    for (int s = 0; s < this_no_of_sets; ++s)
        for (uint32_t w = 0; w < this_assoc; ++w)
            my_lru[s * this_assoc + w] = w;

    for (uint32_t i = 0; i < this_pref_n; ++i) {
        my_sb_valid[i] = false;
        my_sb_top[i] = 0;
        my_sb_rec[i] = this_pref_n - i;
    }
    for (uint32_t i = 0; i < this_pref_n * this_pref_m; ++i) {
        my_sb_blocks[i] = 0;
    }

    for (int i = 0; i < total_no_of_ref; ++i)
    {
        l1_cache(my_tags, my_dirty, my_valid, my_dirty_addr, my_lru,
                 nullptr, nullptr, nullptr, nullptr, nullptr,
                 my_counters, dummy_l2_counters,
                 this_blocksize, this_size, this_assoc,
                 trace_addr[i], trace_rw[i], 0, 0,
                 my_sb_valid, my_sb_top, my_sb_rec, my_sb_blocks,
                 this_pref_n, this_pref_m, 0, 0);
    }

    counters[tid] = my_counters;
}

int main(int argc, char *argv[])
{
	FILE *fp;
	char *trace_file;
	char rw;
	uint32_t addr;

	if (argc != 2)
	{
	    printf("Error: Expected 1 command-line argument (trace file) but was provided %d.\n", (argc - 1));
	    exit(EXIT_FAILURE);
	}

	trace_file = argv[1];

	int assoc_values[6] = {1, 2, 4, 8, 16, 32};
	int size_values[6] = {1024, 2048, 4096, 8192, 16384, 32768};
	int blocksize_values[3] = {16, 32, 64};
	int pref_n_values[3] = {0, 4, 8};
	int pref_m_values[2] = {2, 4};
	int num_assoc = 6;
	int num_size = 6;
	int num_blocksize = 3;
	int num_pref_n = 3;
	int num_pref_m = 2;
	const int num_configs = num_assoc * num_size * num_blocksize * num_pref_n * num_pref_m;  // 648

	int max_assoc = 32;
	int max_size = 32768;
	int min_blocksize = 16;
	int max_sets = max_size / (min_blocksize * 1);
	int max_pref_n = 8;
	int max_pref_m = 4;

	fp = fopen(trace_file, "r");
	if (fp == (FILE *)NULL) {
	    printf("Error: Unable to open file %s\n", trace_file);
	    exit(EXIT_FAILURE);
	}
	int total_no_of_ref = 0;
	{
	    char tmp_rw;
	    uint32_t tmp_addr;
	    while (fscanf(fp, "%c %x\n", &tmp_rw, &tmp_addr) == 2)
	    {
	        total_no_of_ref++;
	    }
	}
	rewind(fp);
	char* trace_rw = new char[total_no_of_ref];
	uint32_t* trace_addr = new uint32_t[total_no_of_ref];
	int idx = 0;
	while (fscanf(fp, "%c %x\n", &rw, &addr) == 2)
	{
	    trace_rw[idx] = rw;
	    trace_addr[idx] = addr;
	    idx++;
	}

	printf("===== Simulator configuration =====\n");
	printf("trace_file: %s\n\n", trace_file);

	int *g_l1_arr_valid, *g_l1_tag_storage, *g_l1_arr_lru;
	char *g_l1_arr_dirty;
	uint32_t *g_l1_arr_dirty_addr;
	cache_params *g_counters;
	uint32_t *g_assoc_values, *g_size_values, *g_blocksize_values, *g_pref_n_values, *g_pref_m_values;
	char *g_trace_rw;
	uint32_t *g_trace_addr;
	bool *g_sb_valid;
	int *g_sb_top, *g_sb_rec;
	uint32_t *g_sb_blocks;

	size_t per_thread_size = max_sets * max_assoc;

	cudaMallocManaged(&g_assoc_values, num_assoc * sizeof(uint32_t));
	cudaMallocManaged(&g_size_values, num_size * sizeof(uint32_t));
	cudaMallocManaged(&g_blocksize_values, num_blocksize * sizeof(uint32_t));
	cudaMallocManaged(&g_pref_n_values, num_pref_n * sizeof(uint32_t));
	cudaMallocManaged(&g_pref_m_values, num_pref_m * sizeof(uint32_t));
	for (int i = 0; i < num_assoc; ++i) g_assoc_values[i] = assoc_values[i];
	for (int i = 0; i < num_size; ++i) g_size_values[i] = size_values[i];
	for (int i = 0; i < num_blocksize; ++i) g_blocksize_values[i] = blocksize_values[i];
	for (int i = 0; i < num_pref_n; ++i) g_pref_n_values[i] = pref_n_values[i];
	for (int i = 0; i < num_pref_m; ++i) g_pref_m_values[i] = pref_m_values[i];

	cudaMallocManaged(&g_l1_arr_valid,      (size_t)num_configs * per_thread_size * sizeof(int));
	cudaMallocManaged(&g_l1_tag_storage,    (size_t)num_configs * per_thread_size * sizeof(int));
	cudaMallocManaged(&g_l1_arr_dirty,      (size_t)num_configs * per_thread_size * sizeof(char));
	cudaMallocManaged(&g_l1_arr_dirty_addr, (size_t)num_configs * per_thread_size * sizeof(uint32_t));
	cudaMallocManaged(&g_l1_arr_lru,        (size_t)num_configs * per_thread_size * sizeof(int));
	cudaMallocManaged(&g_counters,          num_configs * sizeof(cache_params));
	cudaMallocManaged(&g_trace_rw,          total_no_of_ref * sizeof(char));
	cudaMallocManaged(&g_trace_addr,        total_no_of_ref * sizeof(uint32_t));

	cudaMallocManaged(&g_sb_valid,  (size_t)num_configs * max_pref_n * sizeof(bool));
	cudaMallocManaged(&g_sb_top,    (size_t)num_configs * max_pref_n * sizeof(int));
	cudaMallocManaged(&g_sb_rec,    (size_t)num_configs * max_pref_n * sizeof(int));
	cudaMallocManaged(&g_sb_blocks, (size_t)num_configs * max_pref_n * max_pref_m * sizeof(uint32_t));

	for (int i = 0; i < total_no_of_ref; ++i) { g_trace_rw[i] = trace_rw[i]; g_trace_addr[i] = trace_addr[i]; }

	memset(g_l1_arr_valid, 0, (size_t)num_configs * per_thread_size * sizeof(int));
	memset(g_l1_arr_dirty, ' ', (size_t)num_configs * per_thread_size * sizeof(char));
	memset(g_l1_arr_dirty_addr, 0, (size_t)num_configs * per_thread_size * sizeof(uint32_t));

	cudaEvent_t start, stop;
	cudaEventCreate(&start);
	cudaEventCreate(&stop);

	int threadsPerBlock = 32;
	int numBlocks = (num_configs + threadsPerBlock - 1) / threadsPerBlock;

	cudaEventRecord(start);
	sweep_kernel<<<numBlocks, threadsPerBlock>>>(num_configs, num_size, num_blocksize, num_pref_n, num_pref_m,
	                                  max_sets, max_assoc, max_pref_n, max_pref_m,
	                                  g_assoc_values, g_size_values, g_blocksize_values,
	                                  g_pref_n_values, g_pref_m_values,
	                                  g_l1_arr_valid, g_l1_tag_storage, g_l1_arr_dirty,
	                                  g_l1_arr_dirty_addr, g_l1_arr_lru,
	                                  g_sb_valid, g_sb_top, g_sb_rec, g_sb_blocks,
	                                  g_counters,
	                                  g_trace_rw, g_trace_addr, total_no_of_ref);
	cudaEventRecord(stop);
	cudaEventSynchronize(stop);
	float kernel_ms = 0;
	cudaEventElapsedTime(&kernel_ms, start, stop);
	printf("Kernel execution time: %.3f ms\n", kernel_ms);
	cudaEventDestroy(start);
	cudaEventDestroy(stop);

	for (int i = 0; i < num_configs; ++i) {
	    cache_params &c = g_counters[i];
	    int pref_m_idx    = i % num_pref_m;
	    int pref_n_idx    = (i / num_pref_m) % num_pref_n;
	    int blocksize_idx = (i / (num_pref_m * num_pref_n)) % num_blocksize;
	    int size_idx      = (i / (num_pref_m * num_pref_n * num_blocksize)) % num_size;
	    int assoc_idx     = i / (num_pref_m * num_pref_n * num_blocksize * num_size);

	    uint32_t a  = g_assoc_values[assoc_idx];
	    uint32_t s  = g_size_values[size_idx];
	    uint32_t b  = g_blocksize_values[blocksize_idx];
	    uint32_t pn = g_pref_n_values[pref_n_idx];
	    uint32_t pm = g_pref_m_values[pref_m_idx];

	    int total = c.read_hit + c.write_hit + c.read_miss + c.write_miss + c.sb_read_hits + c.sb_write_hits;
	    if (total == 0) {
	        printf("ASSOC=%2u SIZE=%6u BLK=%2u PREF_N=%u PREF_M=%u -> skipped (invalid: fewer than 1 set)\n", a, s, b, pn, pm);
	    } else {
	        double miss_rate = static_cast<double>(c.write_miss + c.read_miss) / total;
	        printf("ASSOC=%2u SIZE=%6u BLK=%2u PREF_N=%u PREF_M=%u -> L1 miss rate = %.4f\n", a, s, b, pn, pm, miss_rate);
	    }
	}

	delete[] trace_rw;
	delete[] trace_addr;
	cudaFree(g_l1_arr_valid);
	cudaFree(g_l1_tag_storage);
	cudaFree(g_l1_arr_dirty);
	cudaFree(g_l1_arr_dirty_addr);
	cudaFree(g_l1_arr_lru);
	cudaFree(g_counters);
	cudaFree(g_assoc_values);
	cudaFree(g_size_values);
	cudaFree(g_blocksize_values);
	cudaFree(g_pref_n_values);
	cudaFree(g_pref_m_values);
	cudaFree(g_trace_rw);
	cudaFree(g_trace_addr);
	cudaFree(g_sb_valid);
	cudaFree(g_sb_top);
	cudaFree(g_sb_rec);
	cudaFree(g_sb_blocks);

	return 0;
}

Overwriting cache_sim_gpu.cu


In [11]:
!nvcc --expt-relaxed-constexpr cache_sim_gpu.cu -o cache_sim_gpu_cuda
!./cache_sim_gpu_cuda gcc_trace.txt

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
===== Simulator configuration =====
trace_file: gcc_trace.txt

Kernel execution time: 2559.288 ms
ASSOC= 1 SIZE=  1024 BLK=16 PREF_N=0 PREF_M=2 -> L1 miss rate = 0.1922
ASSOC= 1 SIZE=  1024 BLK=16 PREF_N=0 PREF_M=4 -> L1 miss rate = 0.1922
ASSOC= 1 SIZE=  1024 BLK=16 PREF_N=4 PREF_M=2 -> L1 miss rate = 0.1436
ASSOC= 1 SIZE=  1024 BLK=16 PREF_N=4 PREF_M=4 -> L1 miss rate = 0.1429
ASSOC= 1 SIZE=  1024 BLK=16 PREF_N=8 PREF_M=2 -> L1 miss rate = 0.1369
ASSOC= 1 SIZE=  1024 BLK=16 PREF_N=8 PREF_M=4 -> L1 miss rate = 0.1349
ASSOC= 1 SIZE=  1024 BLK=32 PREF_N=0 PREF_M=2 -> L1 miss rate = 0.1935
ASSOC= 1 SIZE=  1024 BLK=32 PREF_N=0 PREF_M=4 -> L1 miss rate = 0.1935
ASSOC= 1 SIZE=  1024 BLK=32 PREF_N=4 PREF_M=2 -> L1 miss rate = 0.1639
ASSOC= 1 SIZE=  1024 BLK=32 PREF_N=4 PREF_M=4 -> L1 miss rate = 0.1620
ASSOC